In [1]:
pip install torch-geometric

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip3.10 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
import os

DATA_DIR = "/home/user/ARYA_ANOMALY_DETECTION/anomaly_dataset-20260606T092226Z-3-001/anomaly_dataset"
print("Exists:", os.path.exists(DATA_DIR))
print("Is directory:", os.path.isdir(DATA_DIR))

if os.path.exists(DATA_DIR):
    print(os.listdir(DATA_DIR)[:10])

Exists: True
Is directory: True
['machine-2-8_test_combined.csv', 'machine-2-5_test_combined.csv', 'machine-1-7_test_combined.csv', 'machine-1-8_test_combined.csv', 'machine-2-6_test_combined.csv', 'machine-2-1_test_combined.csv', 'machine-1-1_test_combined.csv', 'machine-2-3_test_combined.csv', 'machine-1-3_test_combined.csv', 'machine-2-7_test_combined.csv']


In [12]:
# graph_builder.py
"""
Graph construction for FedRIVER — SMD Dataset
----------------------------------------------
Strategy: Adaptive Mutual KNN + RBF edge weights

Each row (timestep) becomes a node.
Node features  : 38 sensor readings (already normalized to [0,1]).
Node labels    : binary anomaly label (0 = normal, 1 = anomaly).
Edges          : mutual k-nearest neighbors in Euclidean feature space.
Edge weights   : RBF kernel  w = exp(-d² / σ²)
                 where σ = median of all kNN distances (adaptive per machine).
Fallback       : isolated nodes (no mutual neighbors) are connected to
                 their single nearest neighbor to guarantee connectivity.

Why this is better than your original k=2 cosine approach:
  - k=7 gives ~4x more edges → richer neighborhood signal
  - Euclidean distance is more appropriate than cosine for
    normalized sensor data where magnitude differences matter
  - RBF weights encode how similar two timesteps are, not just
    whether they are neighbors
  - Mutual KNN keeps the graph symmetric and sparse
  - Adaptive σ (per machine) handles different machines having
    different feature spread

Usage:
    from graph_builder import build_graph
    data = build_graph('path/to/machine-1-1_test_combined.csv', k=7, train_ratio=0.6)
    # data.x          : [N, 38] float tensor
    # data.edge_index : [2, E] long tensor
    # data.edge_attr  : [E]    float tensor (RBF weights)
    # data.y          : [N]    long tensor  (0/1)
    # data.train_mask : [N]    bool tensor
"""

import os
import numpy as np
import pandas as pd
import torch
from torch_geometric.data import Data
from sklearn.neighbors import BallTree


# ─────────────────────────────────────────────
#  Core graph builder
# ─────────────────────────────────────────────

def build_graph(
    csv_path: str,
    k: int = 7,
    train_ratio: float = 0.6,
    seed: int = 42,
) -> Data:
    """
    Build a PyG Data object from a single machine CSV file.

    Parameters
    ----------
    csv_path    : path to machine CSV  (38 features + 'label' column)
    k           : number of nearest neighbors  (default 7)
    train_ratio : fraction of NORMAL nodes used for training (default 0.6)
    seed        : random seed for reproducible train/test split

    Returns
    -------
    torch_geometric.data.Data
    """
    # ── 1. Load CSV ──────────────────────────────────────────────────────────
    df = pd.read_csv(csv_path, header=0)

    feature_cols = [c for c in df.columns if c != 'label']
    X = df[feature_cols].values.astype(np.float32)   # [N, 38]
    y = df['label'].values.astype(np.float32)         # [N]
    N = len(X)

    # ── 2. KNN via BallTree (memory-efficient, no full distance matrix) ──────
    tree = BallTree(X, metric='euclidean')
    dists, inds = tree.query(X, k=k + 1)   # +1: first result is self

    dists = dists[:, 1:].astype(np.float32)   # [N, k]  exclude self
    inds  = inds[:, 1:]                        # [N, k]

    # ── 3. Adaptive sigma (median of all kNN distances for this machine) ─────
    sigma = float(np.median(dists))
    if sigma < 1e-8:
        sigma = 1.0   # safety fallback

    # ── 4. RBF weights ────────────────────────────────────────────────────────
    rbf = np.exp(-(dists ** 2) / (sigma ** 2))   # [N, k]

    # ── 5. Build mutual KNN edges ─────────────────────────────────────────────
    # neighbor_set[i] = set of k nearest neighbors of node i
    neighbor_set = [set(inds[i].tolist()) for i in range(N)]

    edge_dict = {}   # (i, j) with i < j  →  weight

    for i in range(N):
        for j_pos, j in enumerate(inds[i]):
            j = int(j)
            if i in neighbor_set[j]:          # mutual: j also has i as neighbor
                key = (min(i, j), max(i, j))
                w   = float(rbf[i, j_pos])
                # keep the higher weight if edge already seen from other side
                if key not in edge_dict or w > edge_dict[key]:
                    edge_dict[key] = w

    # ── 6. Fallback: connect isolated nodes to their nearest neighbor ─────────
    has_edge = set()
    for (i, j) in edge_dict:
        has_edge.add(i)
        has_edge.add(j)

    for i in range(N):
        if i not in has_edge:
            j = int(inds[i, 0])
            w = float(rbf[i, 0])
            key = (min(i, j), max(i, j))
            edge_dict[key] = w

    # ── 7. Convert to directed edge_index (both directions) for PyG ──────────
    src_list = [i for (i, j) in edge_dict] + [j for (i, j) in edge_dict]
    dst_list = [j for (i, j) in edge_dict] + [i for (i, j) in edge_dict]
    w_list   = list(edge_dict.values()) + list(edge_dict.values())

    edge_index = torch.tensor([src_list, dst_list], dtype=torch.long)   # [2, E]
    edge_attr  = torch.tensor(w_list, dtype=torch.float)                 # [E]

    # ── 8. Train mask  ────────────────────────────────────────────────────────
    # Train on a random subset of NORMAL nodes only.
    # Test on ALL remaining nodes (normal + anomaly).
    rng = np.random.default_rng(seed)
    normal_idx = np.where(y == 0)[0]
    n_train    = int(len(normal_idx) * train_ratio)
    train_idx  = rng.choice(normal_idx, size=n_train, replace=False)

    train_mask = torch.zeros(N, dtype=torch.bool)
    train_mask[train_idx] = True

    # ── 9. Pack into PyG Data object ─────────────────────────────────────────
    data = Data(
        x          = torch.tensor(X, dtype=torch.float),
        edge_index = edge_index,
        edge_attr  = edge_attr,
        y          = torch.tensor(y, dtype=torch.long),
        train_mask = train_mask,
    )

    return data


# ─────────────────────────────────────────────
#  Batch builder — builds all 28 machines
# ─────────────────────────────────────────────

def build_all_graphs(
    data_dir: str,
    k: int = 7,
    train_ratio: float = 0.6,
    seed: int = 42,
    verbose: bool = True,
) -> dict:
    """
    Build graphs for all machines in data_dir.

    Returns
    -------
    dict: { machine_id (str) -> torch_geometric.data.Data }
    Example keys: '1-1', '1-2', ..., '3-11'
    """
    csv_files = sorted([
        f for f in os.listdir(data_dir)
        if f.endswith('.csv')
    ])

    graphs = {}

    for fname in csv_files:
        machine_id = (
            fname
            .replace('machine-', '')
            .replace('_test_combined.csv', '')
        )
        csv_path = os.path.join(data_dir, fname)

        if verbose:
            print(f'  Building graph for machine {machine_id} ...', end=' ')

        data = build_graph(csv_path, k=k, train_ratio=train_ratio, seed=seed)

        if verbose:
            n_edges = data.edge_index.shape[1] // 2   # undirected count
            n_anom  = int((data.y != 0).sum())
            print(
                f'nodes={data.num_nodes:6d}  '
                f'edges={n_edges:7d}  '
                f'anomalies={n_anom:5d} '
                f'({100*n_anom/data.num_nodes:.1f}%)'
            )

        graphs[machine_id] = data

    return graphs


# ─────────────────────────────────────────────
#  Quick smoke-test
# ─────────────────────────────────────────────

if __name__ == '__main__':
    import sys

    DATA_DIR = "/home/user/ARYA_ANOMALY_DETECTION/anomaly_dataset-20260606T092226Z-3-001/anomaly_dataset"

    print(f'\nBuilding graphs from: {DATA_DIR}')
    print(f'k=7, train_ratio=0.6, sigma=adaptive (per machine)\n')

    graphs = build_all_graphs(DATA_DIR, k=7, train_ratio=0.6, verbose=True)

    print(f'\nDone. Total machines: {len(graphs)}')

    # Print one sample
    sample = graphs['1-1']
    print(f'\nSample graph (machine 1-1):')
    print(f'  x.shape       : {sample.x.shape}')
    print(f'  edge_index    : {sample.edge_index.shape}')
    print(f'  edge_attr     : {sample.edge_attr.shape}')
    print(f'  y.shape       : {sample.y.shape}')
    print(f'  train nodes   : {sample.train_mask.sum().item()}')
    print(f'  test  nodes   : {(~sample.train_mask).sum().item()}')
    print(f'  anomaly nodes : {(sample.y != 0).sum().item()}')
    print(f'  edge_attr min : {sample.edge_attr.min():.4f}')
    print(f'  edge_attr max : {sample.edge_attr.max():.4f}')




Building graphs from: /home/user/ARYA_ANOMALY_DETECTION/anomaly_dataset-20260606T092226Z-3-001/anomaly_dataset
k=7, train_ratio=0.6, sigma=adaptive (per machine)

  Building graph for machine 1-1 ... nodes= 28479  edges=  60437  anomalies= 2694 (9.5%)
  Building graph for machine 1-2 ... nodes= 23694  edges=  50573  anomalies=  542 (2.3%)
  Building graph for machine 1-3 ... nodes= 23703  edges=  52650  anomalies=  817 (3.4%)
  Building graph for machine 1-4 ... nodes= 23707  edges=  51403  anomalies=  720 (3.0%)
  Building graph for machine 1-5 ... nodes= 23706  edges=  49750  anomalies=  100 (0.4%)
  Building graph for machine 1-6 ... nodes= 23689  edges=  50707  anomalies= 3708 (15.7%)
  Building graph for machine 1-7 ... nodes= 23697  edges=  58016  anomalies= 2398 (10.1%)
  Building graph for machine 1-8 ... nodes= 23699  edges=  49921  anomalies=  763 (3.2%)
  Building graph for machine 2-1 ... nodes= 23694  edges=  47036  anomalies= 1170 (4.9%)
  Building graph for machine 2-2 

In [14]:
import os

DATA_DIR = "/home/user/ARYA_ANOMALY_DETECTION/anomaly_dataset-20260606T092226Z-3-001/anomaly_dataset"

print(os.listdir(DATA_DIR))

['machine-2-8_test_combined.csv', 'machine-2-5_test_combined.csv', 'machine-1-7_test_combined.csv', 'machine-1-8_test_combined.csv', 'machine-3-1_test_combined.csv', 'machine-2-6_test_combined.csv', 'machine-3-10_test_combined.csv', 'machine-2-1_test_combined.csv', 'machine-1-1_test_combined.csv', 'machine-2-3_test_combined.csv', 'machine-1-3_test_combined.csv', 'machine-2-7_test_combined.csv', 'machine-3-11_test_combined.csv', 'machine-1-2_test_combined.csv', 'machine-2-2_test_combined.csv', 'machine-3-5_test_combined.csv', 'machine-3-6_test_combined.csv', 'machine-1-5_test_combined.csv', 'machine-3-7_test_combined.csv', 'machine-2-4_test_combined.csv', 'machine-3-4_test_combined.csv', 'machine-3-9_test_combined.csv', 'machine-3-2_test_combined.csv', 'machine-1-4_test_combined.csv', 'machine-3-3_test_combined.csv', 'machine-2-9_test_combined.csv', 'machine-1-6_test_combined.csv', 'machine-3-8_test_combined.csv']


In [15]:
import os

for root, dirs, files in os.walk(DATA_DIR):
    print("\nFOLDER:", root)
    print("FILES:", files[:10])


FOLDER: /home/user/ARYA_ANOMALY_DETECTION/anomaly_dataset-20260606T092226Z-3-001/anomaly_dataset
FILES: ['machine-2-8_test_combined.csv', 'machine-2-5_test_combined.csv', 'machine-1-7_test_combined.csv', 'machine-1-8_test_combined.csv', 'machine-3-1_test_combined.csv', 'machine-2-6_test_combined.csv', 'machine-3-10_test_combined.csv', 'machine-2-1_test_combined.csv', 'machine-1-1_test_combined.csv', 'machine-2-3_test_combined.csv']


In [16]:
# prep_data_lib.py
"""
Data loading library for FedRIVER — SMD Dataset
------------------------------------------------
Provides two main functions used by ALL other scripts:

    get_all_graphs(data_dir)
        → dict of { machine_id -> PyG Data }
          Builds or loads all 28 machine graphs.

    get_client_ids(data_dir)
        → sorted list of machine_id strings
          e.g. ['1-1', '1-2', ..., '3-11']

All graph construction is delegated to graph_builder.py.
This file handles caching, splitting into FL clients,
and providing a consistent interface to every training script.

Caching
-------
Graphs are expensive to build (BallTree KNN on ~25k nodes).
On first call, built graphs are saved to  data_dir/graph_cache/
as .pt files. Subsequent calls load from cache instantly.
Set  force_rebuild=True  to ignore cache.
"""

import os
import torch
from torch_geometric.data import Data
from graph_builder import build_graph, build_all_graphs


# ─────────────────────────────────────────────
#  Constants
# ─────────────────────────────────────────────

FEATURE_DIM  = 38
K_NEIGHBORS  = 7
TRAIN_RATIO  = 0.6
RANDOM_SEED  = 42
CACHE_SUBDIR = 'graph_cache'


# ─────────────────────────────────────────────
#  Cache helpers
# ─────────────────────────────────────────────

def _cache_path(data_dir: str, machine_id: str) -> str:
    cache_dir = os.path.join(data_dir, CACHE_SUBDIR)
    os.makedirs(cache_dir, exist_ok=True)
    return os.path.join(cache_dir, f'{machine_id}.pt')


def _save_graph(data: Data, path: str):
    torch.save(data, path)


def _load_graph(path: str) -> Data:
    return torch.load(path, weights_only=False)


# ─────────────────────────────────────────────
#  Public API
# ─────────────────────────────────────────────

def get_client_ids(data_dir: str) -> list:
    """
    Return sorted list of machine IDs found in data_dir.
    e.g. ['1-1', '1-2', ..., '3-11']
    """
    ids = []
    for fname in os.listdir(data_dir):
        if fname.endswith('_test_combined.csv'):
            mid = fname.replace('machine-', '').replace('_test_combined.csv', '')
            ids.append(mid)
    return sorted(ids)


def get_all_graphs(
    data_dir: str,
    k: int = K_NEIGHBORS,
    train_ratio: float = TRAIN_RATIO,
    seed: int = RANDOM_SEED,
    force_rebuild: bool = False,
    verbose: bool = True,
) -> dict:
    """
    Build (or load from cache) graphs for all machines.

    Parameters
    ----------
    data_dir      : directory containing machine CSVs
    k             : KNN neighbors for graph construction
    train_ratio   : fraction of normal nodes used for training
    seed          : random seed
    force_rebuild : if True, ignore cache and rebuild
    verbose       : print progress

    Returns
    -------
    dict: { machine_id (str) -> torch_geometric.data.Data }
    """
    client_ids = get_client_ids(data_dir)

    if verbose:
        print(f'Found {len(client_ids)} machines in {data_dir}')

    graphs = {}

    for mid in client_ids:
        cache = _cache_path(data_dir, mid)

        if not force_rebuild and os.path.exists(cache):
            if verbose:
                print(f'  [cache] Loading machine {mid}')
            graphs[mid] = _load_graph(cache)
            continue

        csv_path = os.path.join(
            data_dir, f'machine-{mid}_test_combined.csv'
        )

        if verbose:
            print(f'  [build] Building graph for machine {mid} ...', end=' ')

        data = build_graph(
            csv_path,
            k=k,
            train_ratio=train_ratio,
            seed=seed,
        )

        _save_graph(data, cache)

        if verbose:
            n_edges = data.edge_index.shape[1] // 2
            n_anom  = int((data.y != 0).sum())
            print(
                f'nodes={data.num_nodes:6d}  '
                f'edges={n_edges:7d}  '
                f'anomalies={n_anom:5d} ({100*n_anom/data.num_nodes:.1f}%)'
            )

        graphs[mid] = data

    if verbose:
        total_nodes = sum(g.num_nodes for g in graphs.values())
        total_edges = sum(g.edge_index.shape[1] // 2 for g in graphs.values())
        total_anom  = sum(int((g.y != 0).sum()) for g in graphs.values())
        print(f'\nDataset summary:')
        print(f'  Machines : {len(graphs)}')
        print(f'  Nodes    : {total_nodes:,}')
        print(f'  Edges    : {total_edges:,}  (undirected)')
        print(f'  Anomalies: {total_anom:,} ({100*total_anom/total_nodes:.1f}%)')

    return graphs


def get_single_graph(
    data_dir: str,
    machine_id: str,
    k: int = K_NEIGHBORS,
    train_ratio: float = TRAIN_RATIO,
    seed: int = RANDOM_SEED,
    force_rebuild: bool = False,
) -> Data:
    """
    Build (or load) graph for a single machine.

    Parameters
    ----------
    machine_id : e.g. '1-1', '2-3', '3-11'
    """
    cache = _cache_path(data_dir, machine_id)

    if not force_rebuild and os.path.exists(cache):
        return _load_graph(cache)

    csv_path = os.path.join(
        data_dir, f'machine-{machine_id}_test_combined.csv'
    )

    data = build_graph(
        csv_path,
        k=k,
        train_ratio=train_ratio,
        seed=seed,
    )

    _save_graph(data, cache)
    return data


# ─────────────────────────────────────────────
#  FL helper — used by all federated scripts
# ─────────────────────────────────────────────

def get_fl_partition(graphs: dict) -> tuple:
    """
    Returns (client_ids, graphs) as parallel lists,
    consistent ordering guaranteed.

    Usage in FL training:
        client_ids, graph_list = get_fl_partition(graphs)
        selected = random.sample(client_ids, k=7)
        for cid in selected:
            data = graphs[cid]
    """
    client_ids = sorted(graphs.keys())
    return client_ids, graphs


# ─────────────────────────────────────────────
#  Smoke test
# ─────────────────────────────────────────────

if __name__ == '__main__':
    import sys

    DATA_DIR = "/home/user/ARYA_ANOMALY_DETECTION/anomaly_dataset-20260606T092226Z-3-001/anomaly_dataset"

    print('=' * 60)
    print('prep_data_lib.py — smoke test')
    print('=' * 60)

    # First call: builds and caches
    graphs = get_all_graphs(DATA_DIR, verbose=True)

    print('\nSecond call (should load from cache):')
    graphs2 = get_all_graphs(DATA_DIR, verbose=True)

    # Check single graph access
    g = graphs['1-1']
    print(f'\nMachine 1-1:')
    print(f'  x          : {g.x.shape}')
    print(f'  edge_index : {g.edge_index.shape}')
    print(f'  edge_attr  : {g.edge_attr.shape}')
    print(f'  y          : {g.y.shape}')
    print(f'  train_mask : {g.train_mask.sum().item()} train nodes')
    print(f'  test nodes : {(~g.train_mask).sum().item()}')

    # FL partition
    client_ids, _ = get_fl_partition(graphs)
    print(f'\nFL clients ({len(client_ids)}): {client_ids}')



prep_data_lib.py — smoke test
Found 28 machines in /home/user/ARYA_ANOMALY_DETECTION/anomaly_dataset-20260606T092226Z-3-001/anomaly_dataset
  [build] Building graph for machine 1-1 ... nodes= 28479  edges=  60437  anomalies= 2694 (9.5%)
  [build] Building graph for machine 1-2 ... nodes= 23694  edges=  50573  anomalies=  542 (2.3%)
  [build] Building graph for machine 1-3 ... nodes= 23703  edges=  52650  anomalies=  817 (3.4%)
  [build] Building graph for machine 1-4 ... nodes= 23707  edges=  51403  anomalies=  720 (3.0%)
  [build] Building graph for machine 1-5 ... nodes= 23706  edges=  49750  anomalies=  100 (0.4%)
  [build] Building graph for machine 1-6 ... nodes= 23689  edges=  50707  anomalies= 3708 (15.7%)
  [build] Building graph for machine 1-7 ... nodes= 23697  edges=  58016  anomalies= 2398 (10.1%)
  [build] Building graph for machine 1-8 ... nodes= 23699  edges=  49921  anomalies=  763 (3.2%)
  [build] Building graph for machine 2-1 ... nodes= 23694  edges=  47036  anomalie

In [21]:
import sys
sys.path.append("/home/user/ARYA_ANOMALY_DETECTION")

import prep_data_lib

print("prep_data_lib imported successfully")

prep_data_lib imported successfully


In [19]:
from prep_data_lib import get_all_graphs

DATA_DIR = "/home/user/ARYA_ANOMALY_DETECTION/anomaly_dataset-20260606T092226Z-3-001/anomaly_dataset"

graphs = get_all_graphs(
    DATA_DIR,
    force_rebuild=True,
    verbose=True
)

print("Machines:", len(graphs))

Found 28 machines in /home/user/ARYA_ANOMALY_DETECTION/anomaly_dataset-20260606T092226Z-3-001/anomaly_dataset
  [build] Building graph for machine 1-1 ... nodes= 28479  edges=  60437  anomalies= 2694 (9.5%)
  [build] Building graph for machine 1-2 ... 

KeyboardInterrupt: 

In [24]:
# centralized_ae.py
"""
Centralized Autoencoder Baseline — FedRIVER
--------------------------------------------
Trains a single AE on ALL machines simultaneously.
No federation, no graph structure.
This is the weakest baseline — sets the lower bound.

Key fixes over original code:
  - Threshold computed from TRAINING errors only (no leakage)
  - Extreme error amplification applied consistently
  - Weighted F1 computed correctly across all machines
  - Reproducible via fixed seed

Run:
    python centralized_ae.py --data_dir ./dataset
"""

import os
import sys
import argparse
import random
import copy

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score

from prep_data_lib import get_all_graphs
from models import Autoencoder


# ─────────────────────────────────────────────
#  Config
# ─────────────────────────────────────────────

SEED         = 42
EPOCHS       = 50
LR           = 0.01
EXTREME_P    = 0.05
EXTREME_ALPHA= 3.0
THRESHOLD_Q  = 0.95


# ─────────────────────────────────────────────
#  Reproducibility
# ─────────────────────────────────────────────

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


# ─────────────────────────────────────────────
#  Extreme error amplification
# ─────────────────────────────────────────────

def extreme_error(err: torch.Tensor, p: float = 0.05, alpha: float = 3.0):
    """Amplify top-p fraction of reconstruction errors by alpha."""
    q = torch.quantile(err, 1 - p)
    return torch.where(err > q, err * alpha, err)


# ─────────────────────────────────────────────
#  Train one epoch
# ─────────────────────────────────────────────

def train_one_epoch(model, data, optimizer, loss_fn, device):
    model.train()
    optimizer.zero_grad()

    x = data.x[data.train_mask].to(device)
    x_hat, _ = model(x)

    loss = loss_fn(x_hat, x)
    loss.backward()
    optimizer.step()
    return loss.item()


# ─────────────────────────────────────────────
#  Threshold (train errors only — NO leakage)
# ─────────────────────────────────────────────

def compute_threshold(model, graphs: dict, device, quantile: float = 0.95):
    """
    Compute anomaly threshold from TRAINING node errors only.
    This is the correct approach — test data must not influence threshold.
    """
    model.eval()
    all_errors = []

    with torch.no_grad():
        for data in graphs.values():
            data  = data.to(device)
            x     = data.x[data.train_mask]
            x_hat, _ = model(x)

            err = torch.mean((x_hat - x) ** 2, dim=1)
            err = extreme_error(err, p=EXTREME_P, alpha=EXTREME_ALPHA)
            all_errors.append(err.cpu().numpy())

    all_errors = np.concatenate(all_errors)
    return float(np.quantile(all_errors, quantile))


# ─────────────────────────────────────────────
#  Evaluation (fixed threshold, test nodes only)
# ─────────────────────────────────────────────

def evaluate(model, graphs: dict, threshold: float, device):
    """
    Evaluate on test nodes across all machines.
    Returns node-count-weighted micro F1.
    """
    model.eval()
    total_nodes  = 0
    weighted_f1  = 0.0

    with torch.no_grad():
        for data in graphs.values():
            data  = data.to(device)
            x_hat, _ = model(data.x)

            err = torch.mean((x_hat - data.x) ** 2, dim=1)
            err = extreme_error(err, p=EXTREME_P, alpha=EXTREME_ALPHA)

            test_mask  = ~data.train_mask
            test_err   = err[test_mask].cpu().numpy()
            test_labels= data.y[test_mask].cpu().numpy()

            y_pred = (test_err   > threshold).astype(int)
            y_true = (test_labels != 0).astype(int)

            f1 = f1_score(y_true, y_pred, average='micro', zero_division=0)

            weighted_f1  += data.num_nodes * f1
            total_nodes  += data.num_nodes

    return weighted_f1 / total_nodes


# ─────────────────────────────────────────────
#  Main
# ─────────────────────────────────────────────

def main(data_dir: str):
    set_seed(SEED)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Device : {device}')
    print(f'Epochs : {EPOCHS}')

    # ── Load data ──────────────────────────────────────────────────────────
    print('\nLoading graphs...')
    graphs = get_all_graphs(data_dir, verbose=False)
    print(f'Machines loaded: {len(graphs)}')

    # ── Model ──────────────────────────────────────────────────────────────
    model     = Autoencoder().to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    loss_fn   = nn.MSELoss()

    # ── Training ───────────────────────────────────────────────────────────
    print('\nTraining Centralized AE...')
    for epoch in range(1, EPOCHS + 1):
        total_loss = 0.0
        for data in graphs.values():
            total_loss += train_one_epoch(model, data, optimizer, loss_fn, device)

        if epoch % 10 == 0:
            print(f'  Epoch {epoch:3d}/{EPOCHS}  loss={total_loss:.4f}')

    # ── Threshold (train data only) ────────────────────────────────────────
    threshold = compute_threshold(model, graphs, device, quantile=THRESHOLD_Q)
    print(f'\nAnomaly threshold (Q{THRESHOLD_Q*100:.0f} of train errors): {threshold:.6f}')

    # ── Evaluation ─────────────────────────────────────────────────────────
    f1 = evaluate(model, graphs, threshold, device)
    print(f'\n{"="*45}')
    print(f'Centralized AE  —  Weighted micro-F1 : {f1:.4f}')
    print(f'{"="*45}')

    return f1


if __name__ == '__main__':
    DATA_DIR = "/home/user/ARYA_ANOMALY_DETECTION/anomaly_dataset-20260606T092226Z-3-001/anomaly_dataset"
    main(DATA_DIR)


Device : cuda
Epochs : 50

Loading graphs...
Machines loaded: 28

Training Centralized AE...
  Epoch  10/50  loss=0.7185
  Epoch  20/50  loss=0.4841
  Epoch  30/50  loss=0.3716
  Epoch  40/50  loss=0.3521
  Epoch  50/50  loss=0.3318

Anomaly threshold (Q95 of train errors): 0.034974

Centralized AE  —  Weighted micro-F1 : 0.8937


In [26]:
# centralized_vae.py
"""
Centralized Variational Autoencoder Baseline — FedRIVER
---------------------------------------------------------
Trains a single VAE on ALL machines simultaneously.
No federation, no graph structure.

For anomaly detection with VAE:
  - At eval time, reparameterization is deterministic (z = mu)
  - Reconstruction error is MSE on the mean path
  - Threshold computed from training node errors only (no leakage)

Run:
    python centralized_vae.py --data_dir ./dataset
"""

import os
import sys
import argparse
import random

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score

from prep_data_lib import get_all_graphs
from models import VariationalAutoencoder, vae_loss


# ─────────────────────────────────────────────
#  Config
# ─────────────────────────────────────────────

SEED          = 42
EPOCHS        = 50
LR            = 0.01
BETA          = 0.5    # KL weight — lower = more focus on reconstruction
EXTREME_P     = 0.05
EXTREME_ALPHA = 3.0
THRESHOLD_Q   = 0.95


# ─────────────────────────────────────────────
#  Reproducibility
# ─────────────────────────────────────────────

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


# ─────────────────────────────────────────────
#  Extreme error amplification
# ─────────────────────────────────────────────

def extreme_error(err: torch.Tensor, p: float = 0.05, alpha: float = 3.0):
    q = torch.quantile(err, 1 - p)
    return torch.where(err > q, err * alpha, err)


# ─────────────────────────────────────────────
#  Train one epoch
# ─────────────────────────────────────────────

def train_one_epoch(model, data, optimizer, device):
    model.train()
    optimizer.zero_grad()

    x = data.x[data.train_mask].to(device)
    x_hat, mu, logvar = model(x)

    loss = vae_loss(x_hat, x, mu, logvar, beta=BETA)
    loss.backward()
    optimizer.step()
    return loss.item()


# ─────────────────────────────────────────────
#  Threshold (train errors only — NO leakage)
# ─────────────────────────────────────────────

def compute_threshold(model, graphs: dict, device, quantile: float = 0.95):
    model.eval()
    all_errors = []

    with torch.no_grad():
        for data in graphs.values():
            data = data.to(device)
            x    = data.x[data.train_mask]

            # deterministic pass: reparameterize returns mu at eval
            x_hat, mu, _ = model(x)

            err = torch.mean((x_hat - x) ** 2, dim=1)
            err = extreme_error(err, p=EXTREME_P, alpha=EXTREME_ALPHA)
            all_errors.append(err.cpu().numpy())

    all_errors = np.concatenate(all_errors)
    return float(np.quantile(all_errors, quantile))


# ─────────────────────────────────────────────
#  Evaluation
# ─────────────────────────────────────────────

def evaluate(model, graphs: dict, threshold: float, device):
    model.eval()
    total_nodes = 0
    weighted_f1 = 0.0

    with torch.no_grad():
        for data in graphs.values():
            data  = data.to(device)
            x_hat, mu, _ = model(data.x)

            err = torch.mean((x_hat - data.x) ** 2, dim=1)
            err = extreme_error(err, p=EXTREME_P, alpha=EXTREME_ALPHA)

            test_mask   = ~data.train_mask
            test_err    = err[test_mask].cpu().numpy()
            test_labels = data.y[test_mask].cpu().numpy()

            y_pred = (test_err    > threshold).astype(int)
            y_true = (test_labels != 0).astype(int)

            f1 = f1_score(y_true, y_pred, average='micro', zero_division=0)

            weighted_f1 += data.num_nodes * f1
            total_nodes += data.num_nodes

    return weighted_f1 / total_nodes


# ─────────────────────────────────────────────
#  Main
# ─────────────────────────────────────────────

def main(data_dir: str):
    set_seed(SEED)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Device : {device}')
    print(f'Epochs : {EPOCHS}')
    print(f'Beta   : {BETA}  (KL weight)')

    # ── Load data ──────────────────────────────────────────────────────────
    print('\nLoading graphs...')
    graphs = get_all_graphs(data_dir, verbose=False)
    print(f'Machines loaded: {len(graphs)}')

    # ── Model ──────────────────────────────────────────────────────────────
    model     = VariationalAutoencoder().to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR)

    # ── Training ───────────────────────────────────────────────────────────
    print('\nTraining Centralized VAE...')
    for epoch in range(1, EPOCHS + 1):
        total_loss = 0.0
        for data in graphs.values():
            total_loss += train_one_epoch(model, data, optimizer, device)

        if epoch % 10 == 0:
            print(f'  Epoch {epoch:3d}/{EPOCHS}  loss={total_loss:.4f}')

    # ── Threshold ──────────────────────────────────────────────────────────
    threshold = compute_threshold(model, graphs, device, quantile=THRESHOLD_Q)
    print(f'\nAnomaly threshold (Q{THRESHOLD_Q*100:.0f} of train errors): {threshold:.6f}')

    # ── Evaluation ─────────────────────────────────────────────────────────
    f1 = evaluate(model, graphs, threshold, device)
    print(f'\n{"="*45}')
    print(f'Centralized VAE  —  Weighted micro-F1 : {f1:.4f}')
    print(f'{"="*45}')

    return f1



if __name__ == '__main__':
    DATA_DIR = "/home/user/ARYA_ANOMALY_DETECTION/anomaly_dataset-20260606T092226Z-3-001/anomaly_dataset"
    main(DATA_DIR)




Device : cuda
Epochs : 50
Beta   : 0.5  (KL weight)

Loading graphs...
Machines loaded: 28

Training Centralized VAE...
  Epoch  10/50  loss=0.8904
  Epoch  20/50  loss=0.8793
  Epoch  30/50  loss=0.8752
  Epoch  40/50  loss=0.8729
  Epoch  50/50  loss=0.8715

Anomaly threshold (Q95 of train errors): 0.103431

Centralized VAE  —  Weighted micro-F1 : 0.8833


In [27]:
# centralized_gae.py
"""
Centralized Graph Autoencoder Baseline — FedRIVER
--------------------------------------------------
Trains a single GAE on ALL machines simultaneously.
No federation. Uses graph structure (GCN encoder).

This is the strongest centralized baseline.
Proves graph structure helps over plain AE.

Key difference from AE:
  - Model takes full PyG Data object (uses edge_index)
  - GCN encoder propagates neighbor information
  - Threshold still computed from train nodes only

Run:
    python centralized_gae.py --data_dir ./dataset
"""

import argparse
import random

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score

from prep_data_lib import get_all_graphs
from models import GraphAutoencoder


# ─────────────────────────────────────────────
#  Config
# ─────────────────────────────────────────────

SEED          = 42
EPOCHS        = 50
LR            = 0.01
EXTREME_P     = 0.05
EXTREME_ALPHA = 3.0
THRESHOLD_Q   = 0.95


# ─────────────────────────────────────────────
#  Reproducibility
# ─────────────────────────────────────────────

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


# ─────────────────────────────────────────────
#  Extreme error amplification
# ─────────────────────────────────────────────

def extreme_error(err: torch.Tensor, p: float = 0.05, alpha: float = 3.0):
    q = torch.quantile(err, 1 - p)
    return torch.where(err > q, err * alpha, err)


# ─────────────────────────────────────────────
#  Train one epoch
# ─────────────────────────────────────────────

def train_one_epoch(model, data, optimizer, loss_fn, device):
    model.train()
    optimizer.zero_grad()

    data  = data.to(device)
    x_rec, _ = model(data)

    loss = loss_fn(
        x_rec[data.train_mask],
        data.x[data.train_mask]
    )
    loss.backward()
    optimizer.step()
    return loss.item()


# ─────────────────────────────────────────────
#  Threshold (train errors only — NO leakage)
# ─────────────────────────────────────────────

def compute_threshold(model, graphs: dict, device, quantile: float = 0.95):
    model.eval()
    all_errors = []

    with torch.no_grad():
        for data in graphs.values():
            data     = data.to(device)
            x_rec, _ = model(data)

            err = torch.mean((x_rec - data.x) ** 2, dim=1)
            err = extreme_error(err, p=EXTREME_P, alpha=EXTREME_ALPHA)

            # Only training nodes for threshold
            train_err = err[data.train_mask]
            all_errors.append(train_err.cpu().numpy())

    all_errors = np.concatenate(all_errors)
    return float(np.quantile(all_errors, quantile))


# ─────────────────────────────────────────────
#  Evaluation
# ─────────────────────────────────────────────

def evaluate(model, graphs: dict, threshold: float, device):
    model.eval()
    total_nodes = 0
    weighted_f1 = 0.0

    with torch.no_grad():
        for data in graphs.values():
            data     = data.to(device)
            x_rec, _ = model(data)

            err = torch.mean((x_rec - data.x) ** 2, dim=1)
            err = extreme_error(err, p=EXTREME_P, alpha=EXTREME_ALPHA)

            test_mask   = ~data.train_mask
            test_err    = err[test_mask].cpu().numpy()
            test_labels = data.y[test_mask].cpu().numpy()

            y_pred = (test_err    > threshold).astype(int)
            y_true = (test_labels != 0).astype(int)

            f1 = f1_score(y_true, y_pred, average='micro', zero_division=0)

            weighted_f1 += data.num_nodes * f1
            total_nodes += data.num_nodes

    return weighted_f1 / total_nodes


# ─────────────────────────────────────────────
#  Main
# ─────────────────────────────────────────────

def main(data_dir: str):
    set_seed(SEED)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Device : {device}')
    print(f'Epochs : {EPOCHS}')

    # ── Load data ──────────────────────────────────────────────────────────
    print('\nLoading graphs...')
    graphs = get_all_graphs(data_dir, verbose=False)
    print(f'Machines loaded: {len(graphs)}')

    # ── Model ──────────────────────────────────────────────────────────────
    model     = GraphAutoencoder().to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    loss_fn   = nn.MSELoss()

    # ── Training ───────────────────────────────────────────────────────────
    print('\nTraining Centralized GAE...')
    for epoch in range(1, EPOCHS + 1):
        total_loss = 0.0
        for data in graphs.values():
            total_loss += train_one_epoch(model, data, optimizer, loss_fn, device)

        if epoch % 10 == 0:
            print(f'  Epoch {epoch:3d}/{EPOCHS}  loss={total_loss:.4f}')

    # ── Threshold ──────────────────────────────────────────────────────────
    threshold = compute_threshold(model, graphs, device, quantile=THRESHOLD_Q)
    print(f'\nAnomaly threshold (Q{THRESHOLD_Q*100:.0f} of train errors): {threshold:.6f}')

    # ── Evaluation ─────────────────────────────────────────────────────────
    f1 = evaluate(model, graphs, threshold, device)
    print(f'\n{"="*45}')
    print(f'Centralized GAE  —  Weighted micro-F1 : {f1:.4f}')
    print(f'{"="*45}')

    return f1

if __name__ == '__main__':
    DATA_DIR = "/home/user/ARYA_ANOMALY_DETECTION/anomaly_dataset-20260606T092226Z-3-001/anomaly_dataset"
    main(DATA_DIR)


Device : cuda
Epochs : 50

Loading graphs...
Machines loaded: 28

Training Centralized GAE...
  Epoch  10/50  loss=0.4640
  Epoch  20/50  loss=0.3533
  Epoch  30/50  loss=0.2733
  Epoch  40/50  loss=0.2913
  Epoch  50/50  loss=0.2550

Anomaly threshold (Q95 of train errors): 0.027487

Centralized GAE  —  Weighted micro-F1 : 0.8900


In [29]:
# fedavg_ae.py
"""
Federated Autoencoder with FedAvg — FedRIVER
---------------------------------------------
Federated learning baseline using plain AE + FedAvg aggregation.
No graph structure.

Each round:
  1. Sample CLIENTS_PER_ROUND clients randomly
  2. Each client trains local AE copy for EPOCHS on its own data
  3. Server averages all local weights (FedAvg)
  4. Global model evaluated on all machines

Key fixes over original code:
  - Threshold from training nodes only (no leakage)
  - Proper client filtering (no hidden dirs)
  - Reproducible seed
  - Round-wise F1 tracked and reported

Run:
    python fedavg_ae.py --data_dir ./dataset
"""

import argparse
import random
import copy

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score

from prep_data_lib import get_all_graphs, get_fl_partition
from models import Autoencoder


# ─────────────────────────────────────────────
#  Config
# ─────────────────────────────────────────────

SEED              = 42
NUM_ROUNDS        = 10
EPOCHS            = 50
LR                = 0.01
CLIENTS_PER_ROUND = 7
EXTREME_P         = 0.05
EXTREME_ALPHA     = 3.0
THRESHOLD_Q       = 0.95


# ─────────────────────────────────────────────
#  Reproducibility
# ─────────────────────────────────────────────

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


# ─────────────────────────────────────────────
#  Helpers
# ─────────────────────────────────────────────

def extreme_error(err, p=0.05, alpha=3.0):
    q = torch.quantile(err, 1 - p)
    return torch.where(err > q, err * alpha, err)


def fedavg(local_states: list) -> dict:
    avg = copy.deepcopy(local_states[0])
    for k in avg:
        for i in range(1, len(local_states)):
            avg[k] += local_states[i][k]
        avg[k] /= len(local_states)
    return avg


def train_local(model, data, device, epochs, lr):
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn   = nn.MSELoss()

    data = data.to(device)
    for _ in range(epochs):
        optimizer.zero_grad()
        x     = data.x[data.train_mask]
        x_hat, _ = model(x)
        loss  = loss_fn(x_hat, x)
        loss.backward()
        optimizer.step()


def compute_threshold(model, graphs, device, quantile=0.95):
    model.eval()
    all_errors = []
    with torch.no_grad():
        for data in graphs.values():
            data = data.to(device)
            x    = data.x[data.train_mask]
            x_hat, _ = model(x)
            err  = torch.mean((x_hat - x) ** 2, dim=1)
            err  = extreme_error(err, EXTREME_P, EXTREME_ALPHA)
            all_errors.append(err.cpu().numpy())
    return float(np.quantile(np.concatenate(all_errors), quantile))


def evaluate(model, graphs, threshold, device):
    model.eval()
    total_nodes = 0
    weighted_f1 = 0.0
    with torch.no_grad():
        for data in graphs.values():
            data = data.to(device)
            x_hat, _ = model(data.x)
            err  = torch.mean((x_hat - data.x) ** 2, dim=1)
            err  = extreme_error(err, EXTREME_P, EXTREME_ALPHA)

            test_mask   = ~data.train_mask
            y_pred = (err[test_mask].cpu().numpy()   > threshold).astype(int)
            y_true = (data.y[test_mask].cpu().numpy() != 0).astype(int)

            f1 = f1_score(y_true, y_pred, average='micro', zero_division=0)
            weighted_f1 += data.num_nodes * f1
            total_nodes += data.num_nodes
    return weighted_f1 / total_nodes


# ─────────────────────────────────────────────
#  Main
# ─────────────────────────────────────────────

def main(data_dir: str):
    set_seed(SEED)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Device           : {device}')
    print(f'Rounds           : {NUM_ROUNDS}')
    print(f'Local epochs     : {EPOCHS}')
    print(f'Clients per round: {CLIENTS_PER_ROUND}')

    print('\nLoading graphs...')
    graphs = get_all_graphs(data_dir, verbose=False)
    client_ids, _ = get_fl_partition(graphs)
    print(f'Total clients: {len(client_ids)}')

    global_model = Autoencoder().to(device)

    f1_per_round = []

    print('\nFederated Training (FedAvg AE)...')
    for rnd in range(1, NUM_ROUNDS + 1):

        # ── Client selection ───────────────────────────────────────────────
        selected = random.sample(client_ids, CLIENTS_PER_ROUND)
        local_states = []

        for cid in selected:
            local_model = copy.deepcopy(global_model)
            train_local(local_model, graphs[cid], device, EPOCHS, LR)
            local_states.append(local_model.state_dict())

        # ── FedAvg aggregation ─────────────────────────────────────────────
        global_model.load_state_dict(fedavg(local_states))

        # ── Evaluate ───────────────────────────────────────────────────────
        threshold = compute_threshold(global_model, graphs, device, THRESHOLD_Q)
        f1        = evaluate(global_model, graphs, threshold, device)
        f1_per_round.append(f1)

        print(f'  Round {rnd:2d}/{NUM_ROUNDS}  '
              f'clients={selected}  '
              f'F1={f1:.4f}')

    best_f1 = max(f1_per_round)
    print(f'\n{"="*45}')
    print(f'FedAvg AE  —  Best weighted micro-F1 : {best_f1:.4f}')
    print(f'             (round {f1_per_round.index(best_f1)+1})')
    print(f'{"="*45}')
    print(f'\nRound-wise F1: {[round(f,4) for f in f1_per_round]}')

    return best_f1



if __name__ == '__main__':
    DATA_DIR = "/home/user/ARYA_ANOMALY_DETECTION/anomaly_dataset-20260606T092226Z-3-001/anomaly_dataset"
    main(DATA_DIR)


Device           : cuda
Rounds           : 10
Local epochs     : 50
Clients per round: 7

Loading graphs...
Total clients: 28

Federated Training (FedAvg AE)...
  Round  1/10  clients=['3-2', '1-4', '1-1', '3-5', '2-1', '1-8', '3-4']  F1=0.8824
  Round  2/10  clients=['1-5', '3-5', '1-4', '3-3', '3-8', '3-1', '1-3']  F1=0.8861
  Round  3/10  clients=['3-10', '2-6', '1-2', '1-1', '1-3', '1-7', '1-8']  F1=0.8815
  Round  4/10  clients=['2-9', '3-11', '1-1', '3-1', '1-7', '3-4', '3-2']  F1=0.8855
  Round  5/10  clients=['3-4', '3-1', '2-6', '1-8', '2-7', '3-10', '2-1']  F1=0.8807
  Round  6/10  clients=['3-7', '1-1', '3-6', '1-6', '3-4', '2-6', '2-3']  F1=0.8805
  Round  7/10  clients=['2-1', '1-5', '1-7', '3-6', '2-3', '1-4', '1-3']  F1=0.8809
  Round  8/10  clients=['2-5', '1-4', '2-4', '3-7', '3-11', '2-1', '1-2']  F1=0.8808
  Round  9/10  clients=['3-5', '2-7', '3-1', '1-4', '2-5', '1-3', '3-7']  F1=0.8838
  Round 10/10  clients=['2-2', '3-8', '3-2', '3-11', '2-4', '3-10', '1-7']  F1=

In [30]:
# fedavg_vae.py
"""
Federated Variational Autoencoder with FedAvg — FedRIVER
---------------------------------------------------------
Federated learning baseline using plain VAE + FedAvg aggregation.
No graph structure.  Compare with fedavg_ae.py to check if the
probabilistic latent space helps in the FL setting.

Each round:
  1. Sample CLIENTS_PER_ROUND clients randomly
  2. Each client trains local VAE copy for EPOCHS on its own data
     (ELBO loss = MSE reconstruction + beta * KL divergence)
  3. Server averages all local weights (FedAvg)
  4. Global model evaluated on all machines with deterministic pass

Run:
    python fedavg_vae.py --data_dir ./dataset
"""

import argparse
import random
import copy

import numpy as np
import torch
import torch.optim as optim
from sklearn.metrics import f1_score

from prep_data_lib import get_all_graphs, get_fl_partition
from models import VariationalAutoencoder, vae_loss


# ─────────────────────────────────────────────
#  Config
# ─────────────────────────────────────────────

SEED              = 42
NUM_ROUNDS        = 10
EPOCHS            = 50
LR                = 0.01
BETA              = 0.5       # KL weight in ELBO
CLIENTS_PER_ROUND = 7
EXTREME_P         = 0.05
EXTREME_ALPHA     = 3.0
THRESHOLD_Q       = 0.95


# ─────────────────────────────────────────────
#  Reproducibility
# ─────────────────────────────────────────────

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


# ─────────────────────────────────────────────
#  Helpers
# ─────────────────────────────────────────────

def extreme_error(err: torch.Tensor, p: float = 0.05, alpha: float = 3.0):
    q = torch.quantile(err, 1 - p)
    return torch.where(err > q, err * alpha, err)


def fedavg(local_states: list) -> dict:
    avg = copy.deepcopy(local_states[0])
    for k in avg:
        for i in range(1, len(local_states)):
            avg[k] += local_states[i][k]
        avg[k] /= len(local_states)
    return avg


def train_local(model, data, device, epochs, lr):
    """Train a local VAE copy for `epochs` steps on a single client."""
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    x = data.x[data.train_mask].to(device)
    for _ in range(epochs):
        optimizer.zero_grad()
        x_hat, mu, logvar = model(x)
        loss = vae_loss(x_hat, x, mu, logvar, beta=BETA)
        loss.backward()
        optimizer.step()


def compute_threshold(model, graphs, device, quantile=0.95):
    """Threshold from TRAINING nodes only — no leakage."""
    model.eval()
    all_errors = []
    with torch.no_grad():
        for data in graphs.values():
            x = data.x[data.train_mask].to(device)
            x_hat, mu, _ = model(x)   # deterministic (eval mode → z = mu)
            err = torch.mean((x_hat - x) ** 2, dim=1)
            err = extreme_error(err, EXTREME_P, EXTREME_ALPHA)
            all_errors.append(err.cpu().numpy())
    return float(np.quantile(np.concatenate(all_errors), quantile))


def evaluate(model, graphs, threshold, device):
    """Node-count-weighted micro-F1 across all machines."""
    model.eval()
    total_nodes = 0
    weighted_f1 = 0.0
    with torch.no_grad():
        for data in graphs.values():
            data = data.to(device)
            x_hat, mu, _ = model(data.x)
            err = torch.mean((x_hat - data.x) ** 2, dim=1)
            err = extreme_error(err, EXTREME_P, EXTREME_ALPHA)

            test_mask = ~data.train_mask
            y_pred = (err[test_mask].cpu().numpy()    > threshold).astype(int)
            y_true = (data.y[test_mask].cpu().numpy() != 0).astype(int)

            f1 = f1_score(y_true, y_pred, average='micro', zero_division=0)
            weighted_f1 += data.num_nodes * f1
            total_nodes += data.num_nodes
    return weighted_f1 / total_nodes


# ─────────────────────────────────────────────
#  Main
# ─────────────────────────────────────────────

def main(data_dir: str):
    set_seed(SEED)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Device           : {device}')
    print(f'Rounds           : {NUM_ROUNDS}')
    print(f'Local epochs     : {EPOCHS}')
    print(f'Clients per round: {CLIENTS_PER_ROUND}')
    print(f'Beta (KL weight) : {BETA}')

    print('\nLoading graphs...')
    graphs = get_all_graphs(data_dir, verbose=False)
    client_ids, _ = get_fl_partition(graphs)
    print(f'Total clients: {len(client_ids)}')

    global_model = VariationalAutoencoder().to(device)

    f1_per_round = []

    print('\nFederated Training (FedAvg VAE)...')
    for rnd in range(1, NUM_ROUNDS + 1):

        selected     = random.sample(client_ids, CLIENTS_PER_ROUND)
        local_states = []

        for cid in selected:
            local_model = copy.deepcopy(global_model)
            train_local(local_model, graphs[cid], device, EPOCHS, LR)
            local_states.append(local_model.state_dict())

        global_model.load_state_dict(fedavg(local_states))

        threshold = compute_threshold(global_model, graphs, device, THRESHOLD_Q)
        f1        = evaluate(global_model, graphs, threshold, device)
        f1_per_round.append(f1)

        print(f'  Round {rnd:2d}/{NUM_ROUNDS}  '
              f'clients={selected}  '
              f'threshold={threshold:.6f}  '
              f'F1={f1:.4f}')

    best_f1 = max(f1_per_round)
    print(f'\n{"="*45}')
    print(f'FedAvg VAE  —  Best weighted micro-F1 : {best_f1:.4f}')
    print(f'              (round {f1_per_round.index(best_f1)+1})')
    print(f'{"="*45}')
    print(f'\nRound-wise F1: {[round(f,4) for f in f1_per_round]}')

    return best_f1


if __name__ == '__main__':
    DATA_DIR = "/home/user/ARYA_ANOMALY_DETECTION/anomaly_dataset-20260606T092226Z-3-001/anomaly_dataset"
    main(DATA_DIR)



Device           : cuda
Rounds           : 10
Local epochs     : 50
Clients per round: 7
Beta (KL weight) : 0.5

Loading graphs...
Total clients: 28

Federated Training (FedAvg VAE)...
  Round  1/10  clients=['3-2', '1-4', '1-1', '3-5', '2-1', '1-8', '3-4']  threshold=0.125437  F1=0.8821
  Round  2/10  clients=['1-5', '3-5', '1-4', '3-3', '3-8', '3-1', '1-3']  threshold=0.116531  F1=0.8837
  Round  3/10  clients=['3-10', '2-6', '1-2', '1-1', '1-3', '1-7', '1-8']  threshold=0.119836  F1=0.8816
  Round  4/10  clients=['2-9', '3-11', '1-1', '3-1', '1-7', '3-4', '3-2']  threshold=0.103531  F1=0.8831
  Round  5/10  clients=['3-4', '3-1', '2-6', '1-8', '2-7', '3-10', '2-1']  threshold=0.116200  F1=0.8824
  Round  6/10  clients=['3-7', '1-1', '3-6', '1-6', '3-4', '2-6', '2-3']  threshold=0.109050  F1=0.8801
  Round  7/10  clients=['2-1', '1-5', '1-7', '3-6', '2-3', '1-4', '1-3']  threshold=0.111007  F1=0.8815
  Round  8/10  clients=['2-5', '1-4', '2-4', '3-7', '3-11', '2-1', '1-2']  threshold

In [31]:
# fedavg_gae.py
"""
Federated Graph Autoencoder with FedAvg — FedRIVER
---------------------------------------------------
Federated learning baseline using GAE + FedAvg aggregation.
Uses graph structure (GCN encoder).  This is the direct FL
counterpart of centralized_gae.py and the immediate predecessor
of the FedRIVER variants.

Expected story:
  FL-AE  < Centralized-GAE < FL-GAE < FedRIVER-GAE

Key difference from fedavg_ae.py:
  - Model takes full PyG Data object (uses edge_index)
  - GCN encoder propagates neighbor information
  - Local training must pass the full Data object, not just x

Run:
    python fedavg_gae.py --data_dir ./dataset
"""

import argparse
import random
import copy

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score

from prep_data_lib import get_all_graphs, get_fl_partition
from models import GraphAutoencoder


# ─────────────────────────────────────────────
#  Config
# ─────────────────────────────────────────────

SEED              = 42
NUM_ROUNDS        = 10
EPOCHS            = 50
LR                = 0.01
CLIENTS_PER_ROUND = 7
EXTREME_P         = 0.05
EXTREME_ALPHA     = 3.0
THRESHOLD_Q       = 0.95


# ─────────────────────────────────────────────
#  Reproducibility
# ─────────────────────────────────────────────

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


# ─────────────────────────────────────────────
#  Helpers
# ─────────────────────────────────────────────

def extreme_error(err: torch.Tensor, p: float = 0.05, alpha: float = 3.0):
    q = torch.quantile(err, 1 - p)
    return torch.where(err > q, err * alpha, err)


def fedavg(local_states: list) -> dict:
    avg = copy.deepcopy(local_states[0])
    for k in avg:
        for i in range(1, len(local_states)):
            avg[k] += local_states[i][k]
        avg[k] /= len(local_states)
    return avg


def train_local(model, data, device, epochs, lr):
    """Train a local GAE copy for `epochs` steps on a single client."""
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn   = nn.MSELoss()

    data = data.to(device)
    for _ in range(epochs):
        optimizer.zero_grad()
        x_rec, _ = model(data)
        # train on train_mask nodes only
        loss = loss_fn(
            x_rec[data.train_mask],
            data.x[data.train_mask]
        )
        loss.backward()
        optimizer.step()


def compute_threshold(model, graphs, device, quantile=0.95):
    """Threshold from TRAINING nodes only — no leakage."""
    model.eval()
    all_errors = []
    with torch.no_grad():
        for data in graphs.values():
            data     = data.to(device)
            x_rec, _ = model(data)
            err = torch.mean((x_rec - data.x) ** 2, dim=1)
            err = extreme_error(err, EXTREME_P, EXTREME_ALPHA)
            train_err = err[data.train_mask]
            all_errors.append(train_err.cpu().numpy())
    return float(np.quantile(np.concatenate(all_errors), quantile))


def evaluate(model, graphs, threshold, device):
    """Node-count-weighted micro-F1 across all machines."""
    model.eval()
    total_nodes = 0
    weighted_f1 = 0.0
    with torch.no_grad():
        for data in graphs.values():
            data     = data.to(device)
            x_rec, _ = model(data)
            err = torch.mean((x_rec - data.x) ** 2, dim=1)
            err = extreme_error(err, EXTREME_P, EXTREME_ALPHA)

            test_mask = ~data.train_mask
            y_pred = (err[test_mask].cpu().numpy()    > threshold).astype(int)
            y_true = (data.y[test_mask].cpu().numpy() != 0).astype(int)

            f1 = f1_score(y_true, y_pred, average='micro', zero_division=0)
            weighted_f1 += data.num_nodes * f1
            total_nodes += data.num_nodes
    return weighted_f1 / total_nodes


# ─────────────────────────────────────────────
#  Main
# ─────────────────────────────────────────────

def main(data_dir: str):
    set_seed(SEED)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Device           : {device}')
    print(f'Rounds           : {NUM_ROUNDS}')
    print(f'Local epochs     : {EPOCHS}')
    print(f'Clients per round: {CLIENTS_PER_ROUND}')

    print('\nLoading graphs...')
    graphs = get_all_graphs(data_dir, verbose=False)
    client_ids, _ = get_fl_partition(graphs)
    print(f'Total clients: {len(client_ids)}')

    global_model = GraphAutoencoder().to(device)

    f1_per_round = []

    print('\nFederated Training (FedAvg GAE)...')
    for rnd in range(1, NUM_ROUNDS + 1):

        selected     = random.sample(client_ids, CLIENTS_PER_ROUND)
        local_states = []

        for cid in selected:
            local_model = copy.deepcopy(global_model)
            train_local(local_model, graphs[cid], device, EPOCHS, LR)
            local_states.append(local_model.state_dict())

        global_model.load_state_dict(fedavg(local_states))

        threshold = compute_threshold(global_model, graphs, device, THRESHOLD_Q)
        f1        = evaluate(global_model, graphs, threshold, device)
        f1_per_round.append(f1)

        print(f'  Round {rnd:2d}/{NUM_ROUNDS}  '
              f'clients={selected}  '
              f'threshold={threshold:.6f}  '
              f'F1={f1:.4f}')

    best_f1 = max(f1_per_round)
    print(f'\n{"="*45}')
    print(f'FedAvg GAE  —  Best weighted micro-F1 : {best_f1:.4f}')
    print(f'              (round {f1_per_round.index(best_f1)+1})')
    print(f'{"="*45}')
    print(f'\nRound-wise F1: {[round(f,4) for f in f1_per_round]}')

    return best_f1


if __name__ == '__main__':
    DATA_DIR = "/home/user/ARYA_ANOMALY_DETECTION/anomaly_dataset-20260606T092226Z-3-001/anomaly_dataset"
    main(DATA_DIR)





Device           : cuda
Rounds           : 10
Local epochs     : 50
Clients per round: 7

Loading graphs...
Total clients: 28

Federated Training (FedAvg GAE)...
  Round  1/10  clients=['3-2', '1-4', '1-1', '3-5', '2-1', '1-8', '3-4']  threshold=0.127074  F1=0.8807
  Round  2/10  clients=['1-5', '3-5', '1-4', '3-3', '3-8', '3-1', '1-3']  threshold=0.072036  F1=0.8819
  Round  3/10  clients=['3-10', '2-6', '1-2', '1-1', '1-3', '1-7', '1-8']  threshold=0.075418  F1=0.8782
  Round  4/10  clients=['2-9', '3-11', '1-1', '3-1', '1-7', '3-4', '3-2']  threshold=0.058396  F1=0.8832
  Round  5/10  clients=['3-4', '3-1', '2-6', '1-8', '2-7', '3-10', '2-1']  threshold=0.072887  F1=0.8784
  Round  6/10  clients=['3-7', '1-1', '3-6', '1-6', '3-4', '2-6', '2-3']  threshold=0.072641  F1=0.8786
  Round  7/10  clients=['2-1', '1-5', '1-7', '3-6', '2-3', '1-4', '1-3']  threshold=0.080352  F1=0.8793
  Round  8/10  clients=['2-5', '1-4', '2-4', '3-7', '3-11', '2-1', '1-2']  threshold=0.062056  F1=0.8790
  

In [32]:
# fedriver_gae_var.py
"""
FedRIVER — Variant A: Variance Regularization + FedComb Server
---------------------------------------------------------------
This is your BEST performing FedRIVER variant (F1 ≈ 0.9206).

Local objective adds a variance regularization term:
    L = mean(node_errors) + λ * var(node_errors)

Rationale:
  Penalizing high variance in per-node reconstruction errors
  forces the model to reconstruct ALL nodes consistently —
  not just the easy ones. Anomalies that the model "ignores"
  (large error but small weight) are forced into the loss.
  This sharpens the decision boundary between normal and anomalous.

Server aggregation — FedComb (momentum-adaptive):
    δ_t   = avg_local_Δ          (mean weight delta)
    Θ_t   = β·Θ_{t-1} + (1-β)·δ_t      (momentum)
    v_t  += Θ_t²                         (second moment)
    w_new = w_global + η·Θ_t / (√v_t + ε)

Run:
    python fedriver_gae_var.py --data_dir ./dataset
"""

import argparse
import random
import copy

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score

from prep_data_lib import get_all_graphs, get_fl_partition
from models import GraphAutoencoder


# ─────────────────────────────────────────────
#  Config
# ─────────────────────────────────────────────

SEED              = 42
NUM_ROUNDS        = 10
EPOCHS            = 50
LR                = 0.01
CLIENTS_PER_ROUND = 7
EXTREME_P         = 0.05
EXTREME_ALPHA     = 3.0
THRESHOLD_Q       = 0.95

# Variance regularisation
LAMBDA_VAR        = 0.5    # weight on variance penalty

# FedComb server hyperparameters
BETA_MOMENTUM     = 0.9    # momentum decay
ETA               = 0.01   # server learning rate
EPSILON           = 1e-8   # numerical stability


# ─────────────────────────────────────────────
#  Reproducibility
# ─────────────────────────────────────────────

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


# ─────────────────────────────────────────────
#  Helpers
# ─────────────────────────────────────────────

def extreme_error(err: torch.Tensor, p: float = 0.05, alpha: float = 3.0):
    q = torch.quantile(err, 1 - p)
    return torch.where(err > q, err * alpha, err)


# ─────────────────────────────────────────────
#  FedComb server aggregation
# ─────────────────────────────────────────────

class FedCombServer:
    """
    Momentum-adaptive server aggregation (FedComb).

    Maintains running momentum Θ and second moment v for each
    weight tensor.  Updates the global model using:
        Θ_t  = β·Θ_{t-1} + (1-β)·δ_t
        v_t += Θ_t²
        w    = w_global + η·Θ_t / (√v_t + ε)
    """

    def __init__(self, global_model, beta=0.9, eta=0.01, eps=1e-8):
        self.beta  = beta
        self.eta   = eta
        self.eps   = eps

        # initialise momentum and second moment buffers
        ref = global_model.state_dict()
        self.momentum = {k: torch.zeros_like(v.float()) for k, v in ref.items()}
        self.v        = {k: torch.zeros_like(v.float()) for k, v in ref.items()}

    def aggregate(self, global_model, local_states: list):
        """Aggregate local states into global model via FedComb."""
        global_state = global_model.state_dict()
        n = len(local_states)

        # mean delta
        delta = {}
        for k in global_state:
            stacked = torch.stack(
                [s[k].float() - global_state[k].float() for s in local_states]
            )
            delta[k] = stacked.mean(dim=0)

        # momentum update
        new_state = {}
        for k in global_state:
            self.momentum[k] = (self.beta * self.momentum[k]
                                + (1 - self.beta) * delta[k])
            self.v[k]       += self.momentum[k] ** 2
            update           = self.eta * self.momentum[k] / (
                torch.sqrt(self.v[k]) + self.eps
            )
            new_state[k] = global_state[k].float() + update

        global_model.load_state_dict(new_state)


# ─────────────────────────────────────────────
#  Local training — variance regularised loss
# ─────────────────────────────────────────────

def train_local(model, data, device, epochs, lr, lambda_var):
    """
    Local GAE training with variance regularization.

    Loss = mean(per-node MSE) + λ * var(per-node MSE)
    """
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    data = data.to(device)
    for _ in range(epochs):
        optimizer.zero_grad()
        x_rec, _ = model(data)

        # per-node reconstruction error on train nodes
        node_err = torch.mean(
            (x_rec[data.train_mask] - data.x[data.train_mask]) ** 2,
            dim=1
        )                                              # [N_train]

        # variance regularisation
        loss = node_err.mean() + lambda_var * node_err.var()

        loss.backward()
        optimizer.step()


# ─────────────────────────────────────────────
#  Threshold & evaluation
# ─────────────────────────────────────────────

def compute_threshold(model, graphs, device, quantile=0.95):
    model.eval()
    all_errors = []
    with torch.no_grad():
        for data in graphs.values():
            data     = data.to(device)
            x_rec, _ = model(data)
            err = torch.mean((x_rec - data.x) ** 2, dim=1)
            err = extreme_error(err, EXTREME_P, EXTREME_ALPHA)
            all_errors.append(err[data.train_mask].cpu().numpy())
    return float(np.quantile(np.concatenate(all_errors), quantile))


def evaluate(model, graphs, threshold, device):
    model.eval()
    total_nodes = 0
    weighted_f1 = 0.0
    with torch.no_grad():
        for data in graphs.values():
            data     = data.to(device)
            x_rec, _ = model(data)
            err = torch.mean((x_rec - data.x) ** 2, dim=1)
            err = extreme_error(err, EXTREME_P, EXTREME_ALPHA)

            test_mask = ~data.train_mask
            y_pred = (err[test_mask].cpu().numpy()    > threshold).astype(int)
            y_true = (data.y[test_mask].cpu().numpy() != 0).astype(int)

            f1 = f1_score(y_true, y_pred, average='micro', zero_division=0)
            weighted_f1 += data.num_nodes * f1
            total_nodes += data.num_nodes
    return weighted_f1 / total_nodes


# ─────────────────────────────────────────────
#  Main
# ─────────────────────────────────────────────

def main(data_dir: str):
    set_seed(SEED)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Device           : {device}')
    print(f'Rounds           : {NUM_ROUNDS}')
    print(f'Local epochs     : {EPOCHS}')
    print(f'Clients per round: {CLIENTS_PER_ROUND}')
    print(f'Lambda (var reg) : {LAMBDA_VAR}')
    print(f'FedComb β        : {BETA_MOMENTUM}  η={ETA}')

    print('\nLoading graphs...')
    graphs = get_all_graphs(data_dir, verbose=False)
    client_ids, _ = get_fl_partition(graphs)
    print(f'Total clients: {len(client_ids)}')

    global_model = GraphAutoencoder().to(device)
    server       = FedCombServer(global_model, beta=BETA_MOMENTUM,
                                 eta=ETA, eps=EPSILON)

    f1_per_round = []

    print('\nFedRIVER (Variant A — Variance Reg + FedComb)...')
    for rnd in range(1, NUM_ROUNDS + 1):

        selected     = random.sample(client_ids, CLIENTS_PER_ROUND)
        local_states = []

        for cid in selected:
            local_model = copy.deepcopy(global_model)
            train_local(local_model, graphs[cid], device, EPOCHS, LR, LAMBDA_VAR)
            local_states.append(local_model.state_dict())

        server.aggregate(global_model, local_states)

        threshold = compute_threshold(global_model, graphs, device, THRESHOLD_Q)
        f1        = evaluate(global_model, graphs, threshold, device)
        f1_per_round.append(f1)

        print(f'  Round {rnd:2d}/{NUM_ROUNDS}  '
              f'clients={selected}  '
              f'threshold={threshold:.6f}  '
              f'F1={f1:.4f}')

    best_f1 = max(f1_per_round)
    print(f'\n{"="*50}')
    print(f'FedRIVER Var-Reg  —  Best weighted micro-F1 : {best_f1:.4f}')
    print(f'                     (round {f1_per_round.index(best_f1)+1})')
    print(f'{"="*50}')
    print(f'\nRound-wise F1: {[round(f,4) for f in f1_per_round]}')

    return best_f1


if __name__ == '__main__':
    DATA_DIR = "/home/user/ARYA_ANOMALY_DETECTION/anomaly_dataset-20260606T092226Z-3-001/anomaly_dataset"
    main(DATA_DIR)


Device           : cuda
Rounds           : 10
Local epochs     : 50
Clients per round: 7
Lambda (var reg) : 0.5
FedComb β        : 0.9  η=0.01

Loading graphs...
Total clients: 28

FedRIVER (Variant A — Variance Reg + FedComb)...
  Round  1/10  clients=['3-2', '1-4', '1-1', '3-5', '2-1', '1-8', '3-4']  threshold=0.238176  F1=0.8809
  Round  2/10  clients=['1-5', '3-5', '1-4', '3-3', '3-8', '3-1', '1-3']  threshold=0.220092  F1=0.8808
  Round  3/10  clients=['3-10', '2-6', '1-2', '1-1', '1-3', '1-7', '1-8']  threshold=0.206810  F1=0.8809
  Round  4/10  clients=['2-9', '3-11', '1-1', '3-1', '1-7', '3-4', '3-2']  threshold=0.195290  F1=0.8810
  Round  5/10  clients=['3-4', '3-1', '2-6', '1-8', '2-7', '3-10', '2-1']  threshold=0.184873  F1=0.8810
  Round  6/10  clients=['3-7', '1-1', '3-6', '1-6', '3-4', '2-6', '2-3']  threshold=0.174301  F1=0.8810
  Round  7/10  clients=['2-1', '1-5', '1-7', '3-6', '2-3', '1-4', '1-3']  threshold=0.162762  F1=0.8808
  Round  8/10  clients=['2-5', '1-4', '

In [33]:
# fedriver_gae_kl.py
"""
FedRIVER — Variant B: Variance Regularization + Fixed KL + FedComb
-------------------------------------------------------------------
Extends Variant A by adding a fixed-weight KL divergence term that
aligns each client's local latent distribution toward the global one.

Local objective:
    L = mean(node_errors)
      + λ · var(node_errors)                    ← variance penalty
      + μ · KL(p_local_latent ‖ p_global_latent) ← distribution alignment

KL is estimated as:
    KL(N(μ_local, σ_local) ‖ N(μ_global, σ_global))
Using the closed-form KL between two Gaussians.

Motivation:
  The KL term encourages clients to stay close to the global latent
  space, preventing drift on heterogeneous machines.  Fixed μ means
  the alignment strength is constant across all rounds.

Compare with Variant A (no KL) to see if distribution alignment helps.
Expected: slight improvement over A from reduced client drift, but
          the fixed μ may over-constrain diverse clients → see Variant C.

Run:
    python fedriver_gae_kl.py --data_dir ./dataset
"""

import argparse
import random
import copy

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.metrics import f1_score

from prep_data_lib import get_all_graphs, get_fl_partition
from models import GraphAutoencoder


# ─────────────────────────────────────────────
#  Config
# ─────────────────────────────────────────────

SEED              = 42
NUM_ROUNDS        = 10
EPOCHS            = 50
LR                = 0.01
CLIENTS_PER_ROUND = 7
EXTREME_P         = 0.05
EXTREME_ALPHA     = 3.0
THRESHOLD_Q       = 0.95

LAMBDA_VAR        = 0.5     # variance regularisation weight
MU_KL             = 0.005   # KL divergence weight (fixed)

BETA_MOMENTUM     = 0.9
ETA               = 0.01
EPSILON           = 1e-8


# ─────────────────────────────────────────────
#  Reproducibility
# ─────────────────────────────────────────────

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


# ─────────────────────────────────────────────
#  KL divergence between two Gaussians
# ─────────────────────────────────────────────

def gaussian_kl(mu1, std1, mu2, std2):
    """
    Closed-form KL(N(mu1,std1²) ‖ N(mu2,std2²)) averaged over nodes.
    All inputs: [N, latent_dim] tensors.
    """
    var1 = std1 ** 2 + 1e-8
    var2 = std2 ** 2 + 1e-8
    kl = (torch.log(std2 / (std1 + 1e-8))
          + (var1 + (mu1 - mu2) ** 2) / (2 * var2)
          - 0.5)
    return kl.mean()


# ─────────────────────────────────────────────
#  FedComb server
# ─────────────────────────────────────────────

class FedCombServer:
    def __init__(self, global_model, beta=0.9, eta=0.01, eps=1e-8):
        self.beta = beta
        self.eta  = eta
        self.eps  = eps
        ref = global_model.state_dict()
        self.momentum = {k: torch.zeros_like(v.float()) for k, v in ref.items()}
        self.v        = {k: torch.zeros_like(v.float()) for k, v in ref.items()}

    def aggregate(self, global_model, local_states):
        global_state = global_model.state_dict()
        delta = {
            k: torch.stack([s[k].float() - global_state[k].float()
                            for s in local_states]).mean(0)
            for k in global_state
        }
        new_state = {}
        for k in global_state:
            self.momentum[k] = self.beta * self.momentum[k] + (1 - self.beta) * delta[k]
            self.v[k]       += self.momentum[k] ** 2
            new_state[k]     = (global_state[k].float()
                                + self.eta * self.momentum[k]
                                / (torch.sqrt(self.v[k]) + self.eps))
        global_model.load_state_dict(new_state)


# ─────────────────────────────────────────────
#  Local training — var + KL loss
# ─────────────────────────────────────────────

def train_local(model, data, device, epochs, lr,
                lambda_var, mu_kl, global_latent_stats):
    """
    Local GAE training with variance reg + fixed KL alignment.

    global_latent_stats: (mu_g, std_g) both [N_train, latent_dim]
                         computed from the global model before the round.
    """
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    data = data.to(device)

    mu_g, std_g = global_latent_stats  # reference distribution

    for _ in range(epochs):
        optimizer.zero_grad()
        x_rec, latent = model(data)   # latent: [N, latent_dim]

        # reconstruction error per node (train nodes)
        node_err = torch.mean(
            (x_rec[data.train_mask] - data.x[data.train_mask]) ** 2,
            dim=1
        )

        # local latent stats for train nodes
        z_train   = latent[data.train_mask]
        mu_local  = z_train.mean(dim=0, keepdim=True).expand_as(z_train)
        std_local = z_train.std(dim=0, keepdim=True).expand_as(z_train) + 1e-8

        kl_term = gaussian_kl(mu_local, std_local, mu_g, std_g)

        loss = (node_err.mean()
                + lambda_var * node_err.var()
                + mu_kl * kl_term)

        loss.backward()
        optimizer.step()


def get_global_latent_stats(model, data, device):
    """Compute (mu, std) of global model latent for train nodes."""
    model.eval()
    with torch.no_grad():
        data = data.to(device)
        _, latent = model(data)
        z     = latent[data.train_mask]
        mu_g  = z.mean(dim=0, keepdim=True).expand_as(z)
        std_g = z.std(dim=0, keepdim=True).expand_as(z) + 1e-8
    return mu_g.detach(), std_g.detach()


# ─────────────────────────────────────────────
#  Threshold & evaluation
# ─────────────────────────────────────────────

def extreme_error(err, p=0.05, alpha=3.0):
    q = torch.quantile(err, 1 - p)
    return torch.where(err > q, err * alpha, err)


def compute_threshold(model, graphs, device, quantile=0.95):
    model.eval()
    all_errors = []
    with torch.no_grad():
        for data in graphs.values():
            data     = data.to(device)
            x_rec, _ = model(data)
            err = torch.mean((x_rec - data.x) ** 2, dim=1)
            err = extreme_error(err, EXTREME_P, EXTREME_ALPHA)
            all_errors.append(err[data.train_mask].cpu().numpy())
    return float(np.quantile(np.concatenate(all_errors), quantile))


def evaluate(model, graphs, threshold, device):
    model.eval()
    total_nodes = 0
    weighted_f1 = 0.0
    with torch.no_grad():
        for data in graphs.values():
            data     = data.to(device)
            x_rec, _ = model(data)
            err = torch.mean((x_rec - data.x) ** 2, dim=1)
            err = extreme_error(err, EXTREME_P, EXTREME_ALPHA)

            test_mask = ~data.train_mask
            y_pred = (err[test_mask].cpu().numpy()    > threshold).astype(int)
            y_true = (data.y[test_mask].cpu().numpy() != 0).astype(int)

            f1 = f1_score(y_true, y_pred, average='micro', zero_division=0)
            weighted_f1 += data.num_nodes * f1
            total_nodes += data.num_nodes
    return weighted_f1 / total_nodes


# ─────────────────────────────────────────────
#  Main
# ─────────────────────────────────────────────

def main(data_dir: str):
    set_seed(SEED)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Device           : {device}')
    print(f'Rounds           : {NUM_ROUNDS}')
    print(f'Local epochs     : {EPOCHS}')
    print(f'Clients per round: {CLIENTS_PER_ROUND}')
    print(f'Lambda (var reg) : {LAMBDA_VAR}')
    print(f'Mu (KL weight)   : {MU_KL}')
    print(f'FedComb β={BETA_MOMENTUM}  η={ETA}')

    print('\nLoading graphs...')
    graphs = get_all_graphs(data_dir, verbose=False)
    client_ids, _ = get_fl_partition(graphs)
    print(f'Total clients: {len(client_ids)}')

    global_model = GraphAutoencoder().to(device)
    server       = FedCombServer(global_model, beta=BETA_MOMENTUM,
                                 eta=ETA, eps=EPSILON)

    f1_per_round = []

    print('\nFedRIVER (Variant B — Var + Fixed KL + FedComb)...')
    for rnd in range(1, NUM_ROUNDS + 1):

        selected     = random.sample(client_ids, CLIENTS_PER_ROUND)
        local_states = []

        for cid in selected:
            local_model = copy.deepcopy(global_model)

            # compute global reference distribution for this client's graph
            g_stats = get_global_latent_stats(global_model, graphs[cid], device)

            train_local(local_model, graphs[cid], device, EPOCHS, LR,
                        LAMBDA_VAR, MU_KL, g_stats)
            local_states.append(local_model.state_dict())

        server.aggregate(global_model, local_states)

        threshold = compute_threshold(global_model, graphs, device, THRESHOLD_Q)
        f1        = evaluate(global_model, graphs, threshold, device)
        f1_per_round.append(f1)

        print(f'  Round {rnd:2d}/{NUM_ROUNDS}  '
              f'clients={selected}  '
              f'threshold={threshold:.6f}  '
              f'F1={f1:.4f}')

    best_f1 = max(f1_per_round)
    print(f'\n{"="*50}')
    print(f'FedRIVER Var+KL  —  Best weighted micro-F1 : {best_f1:.4f}')
    print(f'                    (round {f1_per_round.index(best_f1)+1})')
    print(f'{"="*50}')
    print(f'\nRound-wise F1: {[round(f,4) for f in f1_per_round]}')

    return best_f1


if __name__ == '__main__':
    DATA_DIR = "/home/user/ARYA_ANOMALY_DETECTION/anomaly_dataset-20260606T092226Z-3-001/anomaly_dataset"
    main(DATA_DIR)




Device           : cuda
Rounds           : 10
Local epochs     : 50
Clients per round: 7
Lambda (var reg) : 0.5
Mu (KL weight)   : 0.005
FedComb β=0.9  η=0.01

Loading graphs...
Total clients: 28

FedRIVER (Variant B — Var + Fixed KL + FedComb)...
  Round  1/10  clients=['3-2', '1-4', '1-1', '3-5', '2-1', '1-8', '3-4']  threshold=0.238596  F1=0.8808
  Round  2/10  clients=['1-5', '3-5', '1-4', '3-3', '3-8', '3-1', '1-3']  threshold=0.220029  F1=0.8805
  Round  3/10  clients=['3-10', '2-6', '1-2', '1-1', '1-3', '1-7', '1-8']  threshold=0.205861  F1=0.8806
  Round  4/10  clients=['2-9', '3-11', '1-1', '3-1', '1-7', '3-4', '3-2']  threshold=0.194078  F1=0.8808
  Round  5/10  clients=['3-4', '3-1', '2-6', '1-8', '2-7', '3-10', '2-1']  threshold=0.184357  F1=0.8807
  Round  6/10  clients=['3-7', '1-1', '3-6', '1-6', '3-4', '2-6', '2-3']  threshold=0.176089  F1=0.8805
  Round  7/10  clients=['2-1', '1-5', '1-7', '3-6', '2-3', '1-4', '1-3']  threshold=0.168635  F1=0.8806
  Round  8/10  client

In [34]:
# fedriver_gae_adaptive.py
"""
FedRIVER — Variant C: Adaptive λ·KL (Full Design / FedDual)
------------------------------------------------------------
The complete FedRIVER design: the KL penalty weight is adaptively
computed per client per round based on how much the client's local
performance deviates from the global performance.

Local objective:
    L = mean(node_errors)
      + σ(acc_local - acc_global) · KL(p_local ‖ p_global)

where:
  - σ(·) is the sigmoid function → output in (0, 1)
  - acc_local  = local F1 on train nodes using local model
  - acc_global = local F1 on train nodes using global model
  - KL(p_local ‖ p_global) = KL between local and global latent Gaussians

Intuition:
  If a client diverges from global (acc_local >> acc_global), the KL
  penalty is high → pulls it back.
  If the client is already well-aligned, the KL penalty is near zero
  → local adaptation is unconstrained.

This is the most principled variant but requires estimating per-round
client accuracy, adding overhead compared to Variant A.

Run:
    python fedriver_gae_adaptive.py --data_dir ./dataset
"""

import argparse
import random
import copy

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score

from prep_data_lib import get_all_graphs, get_fl_partition
from models import GraphAutoencoder


# ─────────────────────────────────────────────
#  Config
# ─────────────────────────────────────────────

SEED              = 42
NUM_ROUNDS        = 10
EPOCHS            = 50
LR                = 0.01
CLIENTS_PER_ROUND = 7
EXTREME_P         = 0.05
EXTREME_ALPHA     = 3.0
THRESHOLD_Q       = 0.95

BETA_MOMENTUM     = 0.9
ETA               = 0.01
EPSILON           = 1e-8


# ─────────────────────────────────────────────
#  Reproducibility
# ─────────────────────────────────────────────

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


# ─────────────────────────────────────────────
#  Utility
# ─────────────────────────────────────────────

def extreme_error(err, p=0.05, alpha=3.0):
    q = torch.quantile(err, 1 - p)
    return torch.where(err > q, err * alpha, err)


def sigmoid(x: float) -> float:
    return 1.0 / (1.0 + np.exp(-x))


def gaussian_kl(mu1, std1, mu2, std2):
    """KL(N(mu1,std1²) ‖ N(mu2,std2²)), averaged over elements."""
    var1 = std1 ** 2 + 1e-8
    var2 = std2 ** 2 + 1e-8
    kl = (torch.log(std2 / (std1 + 1e-8))
          + (var1 + (mu1 - mu2) ** 2) / (2 * var2)
          - 0.5)
    return kl.mean()


# ─────────────────────────────────────────────
#  FedComb server
# ─────────────────────────────────────────────

class FedCombServer:
    def __init__(self, global_model, beta=0.9, eta=0.01, eps=1e-8):
        self.beta = beta
        self.eta  = eta
        self.eps  = eps
        ref = global_model.state_dict()
        self.momentum = {k: torch.zeros_like(v.float()) for k, v in ref.items()}
        self.v        = {k: torch.zeros_like(v.float()) for k, v in ref.items()}

    def aggregate(self, global_model, local_states):
        global_state = global_model.state_dict()
        delta = {
            k: torch.stack([s[k].float() - global_state[k].float()
                            for s in local_states]).mean(0)
            for k in global_state
        }
        new_state = {}
        for k in global_state:
            self.momentum[k] = self.beta * self.momentum[k] + (1 - self.beta) * delta[k]
            self.v[k]       += self.momentum[k] ** 2
            new_state[k]     = (global_state[k].float()
                                + self.eta * self.momentum[k]
                                / (torch.sqrt(self.v[k]) + self.eps))
        global_model.load_state_dict(new_state)


# ─────────────────────────────────────────────
#  Client accuracy helper
# ─────────────────────────────────────────────

def client_accuracy(model, data, threshold, device):
    """Micro-F1 on train nodes only (used to compute adaptive λ)."""
    model.eval()
    with torch.no_grad():
        data     = data.to(device)
        x_rec, _ = model(data)
        err = torch.mean((x_rec - data.x) ** 2, dim=1)
        err = extreme_error(err, EXTREME_P, EXTREME_ALPHA)

        train_err  = err[data.train_mask].cpu().numpy()
        train_lbls = data.y[data.train_mask].cpu().numpy()

        y_pred = (train_err    > threshold).astype(int)
        y_true = (train_lbls   != 0).astype(int)
        return f1_score(y_true, y_pred, average='micro', zero_division=0)


# ─────────────────────────────────────────────
#  Latent stats helper
# ─────────────────────────────────────────────

def latent_stats(model, data, device):
    """Mean and std of latent vectors for train nodes."""
    model.eval()
    with torch.no_grad():
        data = data.to(device)
        _, latent = model(data)
        z   = latent[data.train_mask]
        mu  = z.mean(dim=0, keepdim=True).expand_as(z)
        std = z.std(dim=0, keepdim=True).expand_as(z) + 1e-8
    return mu.detach(), std.detach()


# ─────────────────────────────────────────────
#  Local training — adaptive KL
# ─────────────────────────────────────────────

def train_local(model, data, device, epochs, lr,
                lambda_kl_weight, global_latent_stats):
    """
    Local training with adaptive KL penalty.

    lambda_kl_weight: scalar computed as σ(acc_local - acc_global)
    """
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    data = data.to(device)

    mu_g, std_g = global_latent_stats

    for _ in range(epochs):
        optimizer.zero_grad()
        x_rec, latent = model(data)

        node_err = torch.mean(
            (x_rec[data.train_mask] - data.x[data.train_mask]) ** 2,
            dim=1
        )

        z_train   = latent[data.train_mask]
        mu_local  = z_train.mean(dim=0, keepdim=True).expand_as(z_train)
        std_local = z_train.std(dim=0, keepdim=True).expand_as(z_train) + 1e-8

        kl_term = gaussian_kl(mu_local, std_local, mu_g, std_g)

        loss = node_err.mean() + lambda_kl_weight * kl_term
        loss.backward()
        optimizer.step()


# ─────────────────────────────────────────────
#  Threshold & evaluation
# ─────────────────────────────────────────────

def compute_threshold(model, graphs, device, quantile=0.95):
    model.eval()
    all_errors = []
    with torch.no_grad():
        for data in graphs.values():
            data     = data.to(device)
            x_rec, _ = model(data)
            err = torch.mean((x_rec - data.x) ** 2, dim=1)
            err = extreme_error(err, EXTREME_P, EXTREME_ALPHA)
            all_errors.append(err[data.train_mask].cpu().numpy())
    return float(np.quantile(np.concatenate(all_errors), quantile))


def evaluate(model, graphs, threshold, device):
    model.eval()
    total_nodes = 0
    weighted_f1 = 0.0
    with torch.no_grad():
        for data in graphs.values():
            data     = data.to(device)
            x_rec, _ = model(data)
            err = torch.mean((x_rec - data.x) ** 2, dim=1)
            err = extreme_error(err, EXTREME_P, EXTREME_ALPHA)

            test_mask = ~data.train_mask
            y_pred = (err[test_mask].cpu().numpy()    > threshold).astype(int)
            y_true = (data.y[test_mask].cpu().numpy() != 0).astype(int)

            f1 = f1_score(y_true, y_pred, average='micro', zero_division=0)
            weighted_f1 += data.num_nodes * f1
            total_nodes += data.num_nodes
    return weighted_f1 / total_nodes


# ─────────────────────────────────────────────
#  Main
# ─────────────────────────────────────────────

def main(data_dir: str):
    set_seed(SEED)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Device           : {device}')
    print(f'Rounds           : {NUM_ROUNDS}')
    print(f'Local epochs     : {EPOCHS}')
    print(f'Clients per round: {CLIENTS_PER_ROUND}')
    print(f'FedComb β={BETA_MOMENTUM}  η={ETA}')
    print(f'Adaptive λ = σ(acc_local - acc_global)')

    print('\nLoading graphs...')
    graphs = get_all_graphs(data_dir, verbose=False)
    client_ids, _ = get_fl_partition(graphs)
    print(f'Total clients: {len(client_ids)}')

    global_model = GraphAutoencoder().to(device)
    server       = FedCombServer(global_model, beta=BETA_MOMENTUM,
                                 eta=ETA, eps=EPSILON)

    f1_per_round = []

    print('\nFedRIVER (Variant C — Adaptive KL + FedComb)...')
    for rnd in range(1, NUM_ROUNDS + 1):

        selected     = random.sample(client_ids, CLIENTS_PER_ROUND)
        local_states = []

        # compute global threshold once per round for adaptive λ calculation
        global_threshold = compute_threshold(global_model, graphs, device, THRESHOLD_Q)

        for cid in selected:
            local_model = copy.deepcopy(global_model)

            # global model accuracy on this client
            acc_global = client_accuracy(global_model, graphs[cid],
                                         global_threshold, device)

            # local model accuracy after one quick pass (proxy for convergence gap)
            temp_model = copy.deepcopy(global_model)
            temp_model.train()
            opt_tmp = optim.Adam(temp_model.parameters(), lr=LR)
            data_tmp = graphs[cid].to(device)
            opt_tmp.zero_grad()
            x_rec_tmp, _ = temp_model(data_tmp)
            loss_tmp = nn.MSELoss()(
                x_rec_tmp[data_tmp.train_mask],
                data_tmp.x[data_tmp.train_mask]
            )
            loss_tmp.backward()
            opt_tmp.step()
            acc_local = client_accuracy(temp_model, graphs[cid],
                                        global_threshold, device)

            # adaptive λ
            lambda_kl = sigmoid(acc_local - acc_global)

            # get global latent stats for KL reference
            g_stats = latent_stats(global_model, graphs[cid], device)

            train_local(local_model, graphs[cid], device, EPOCHS, LR,
                        lambda_kl, g_stats)
            local_states.append(local_model.state_dict())

        server.aggregate(global_model, local_states)

        threshold = compute_threshold(global_model, graphs, device, THRESHOLD_Q)
        f1        = evaluate(global_model, graphs, threshold, device)
        f1_per_round.append(f1)

        print(f'  Round {rnd:2d}/{NUM_ROUNDS}  '
              f'clients={selected}  '
              f'threshold={threshold:.6f}  '
              f'F1={f1:.4f}')

    best_f1 = max(f1_per_round)
    print(f'\n{"="*55}')
    print(f'FedRIVER Adaptive KL  —  Best weighted micro-F1 : {best_f1:.4f}')
    print(f'                         (round {f1_per_round.index(best_f1)+1})')
    print(f'{"="*55}')
    print(f'\nRound-wise F1: {[round(f,4) for f in f1_per_round]}')

    return best_f1


if __name__ == '__main__':
    DATA_DIR = "/home/user/ARYA_ANOMALY_DETECTION/anomaly_dataset-20260606T092226Z-3-001/anomaly_dataset"
    main(DATA_DIR)




Device           : cuda
Rounds           : 10
Local epochs     : 50
Clients per round: 7
FedComb β=0.9  η=0.01
Adaptive λ = σ(acc_local - acc_global)

Loading graphs...
Total clients: 28

FedRIVER (Variant C — Adaptive KL + FedComb)...
  Round  1/10  clients=['3-2', '1-4', '1-1', '3-5', '2-1', '1-8', '3-4']  threshold=0.238995  F1=0.8809
  Round  2/10  clients=['1-5', '3-5', '1-4', '3-3', '3-8', '3-1', '1-3']  threshold=0.219982  F1=0.8805
  Round  3/10  clients=['3-10', '2-6', '1-2', '1-1', '1-3', '1-7', '1-8']  threshold=0.205530  F1=0.8806
  Round  4/10  clients=['2-9', '3-11', '1-1', '3-1', '1-7', '3-4', '3-2']  threshold=0.193549  F1=0.8807
  Round  5/10  clients=['3-4', '3-1', '2-6', '1-8', '2-7', '3-10', '2-1']  threshold=0.183990  F1=0.8807
  Round  6/10  clients=['3-7', '1-1', '3-6', '1-6', '3-4', '2-6', '2-3']  threshold=0.175787  F1=0.8806
  Round  7/10  clients=['2-1', '1-5', '1-7', '3-6', '2-3', '1-4', '1-3']  threshold=0.168057  F1=0.8805
  Round  8/10  clients=['2-5', '1

In [35]:
# fedavg_ae_dp.py
"""
Federated Autoencoder with Differential Privacy — FedRIVER
-----------------------------------------------------------
FL-AE + FedAvg + Gaussian DP noise injected before aggregation.

DP mechanism (Gaussian mechanism):
  1. Each client clips gradients to L2-norm ≤ CLIP_NORM
  2. Server adds Gaussian noise N(0, (NOISE_MULTIPLIER * CLIP_NORM)²)
     to each averaged parameter before broadcasting

This is the standard DP-FedAvg approach (McMahan et al., 2018).
Expected: DP hurts F1 vs plain FL-AE → motivates non-DP FedRIVER.

Run:
    python fedavg_ae_dp.py --data_dir ./dataset
"""

import argparse
import random
import copy

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score

from prep_data_lib import get_all_graphs, get_fl_partition
from models import Autoencoder


# ─────────────────────────────────────────────
#  Config
# ─────────────────────────────────────────────

SEED              = 42
NUM_ROUNDS        = 10
EPOCHS            = 50
LR                = 0.01
CLIENTS_PER_ROUND = 7
EXTREME_P         = 0.05
EXTREME_ALPHA     = 3.0
THRESHOLD_Q       = 0.95

# DP parameters
CLIP_NORM         = 1.0    # L2 gradient clipping threshold
NOISE_MULTIPLIER  = 0.01   # Gaussian noise std = NOISE_MULTIPLIER * CLIP_NORM


# ─────────────────────────────────────────────
#  Reproducibility
# ─────────────────────────────────────────────

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


# ─────────────────────────────────────────────
#  DP-FedAvg helpers
# ─────────────────────────────────────────────

def extreme_error(err: torch.Tensor, p: float = 0.05, alpha: float = 3.0):
    q = torch.quantile(err, 1 - p)
    return torch.where(err > q, err * alpha, err)


def clip_gradients(model, max_norm: float):
    """Clip ALL model parameters' gradients to L2 norm ≤ max_norm."""
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)


def fedavg_with_dp(local_states: list, clip_norm: float, noise_mult: float) -> dict:
    """
    FedAvg aggregation + Gaussian noise injection (DP mechanism).
    Noise std = noise_mult * clip_norm  (per parameter element).
    """
    avg = copy.deepcopy(local_states[0])
    n   = len(local_states)

    for k in avg:
        for i in range(1, n):
            avg[k] = avg[k].float() + local_states[i][k].float()
        avg[k] = avg[k].float() / n

        # Gaussian noise scaled to sensitivity / n
        noise_std = (noise_mult * clip_norm) / n
        noise     = torch.randn_like(avg[k].float()) * noise_std
        avg[k]    = avg[k].float() + noise

    return avg


def train_local_dp(model, data, device, epochs, lr, clip_norm):
    """Train with per-step gradient clipping."""
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn   = nn.MSELoss()

    data = data.to(device)
    for _ in range(epochs):
        optimizer.zero_grad()
        x     = data.x[data.train_mask]
        x_hat, _ = model(x)
        loss  = loss_fn(x_hat, x)
        loss.backward()
        clip_gradients(model, clip_norm)
        optimizer.step()


def compute_threshold(model, graphs, device, quantile=0.95):
    model.eval()
    all_errors = []
    with torch.no_grad():
        for data in graphs.values():
            data = data.to(device)
            x    = data.x[data.train_mask]
            x_hat, _ = model(x)
            err  = torch.mean((x_hat - x) ** 2, dim=1)
            err  = extreme_error(err, EXTREME_P, EXTREME_ALPHA)
            all_errors.append(err.cpu().numpy())
    return float(np.quantile(np.concatenate(all_errors), quantile))


def evaluate(model, graphs, threshold, device):
    model.eval()
    total_nodes = 0
    weighted_f1 = 0.0
    with torch.no_grad():
        for data in graphs.values():
            data  = data.to(device)
            x_hat, _ = model(data.x)
            err   = torch.mean((x_hat - data.x) ** 2, dim=1)
            err   = extreme_error(err, EXTREME_P, EXTREME_ALPHA)

            test_mask = ~data.train_mask
            y_pred = (err[test_mask].cpu().numpy()    > threshold).astype(int)
            y_true = (data.y[test_mask].cpu().numpy() != 0).astype(int)

            f1 = f1_score(y_true, y_pred, average='micro', zero_division=0)
            weighted_f1 += data.num_nodes * f1
            total_nodes += data.num_nodes
    return weighted_f1 / total_nodes


# ─────────────────────────────────────────────
#  Main
# ─────────────────────────────────────────────

def main(data_dir: str):
    set_seed(SEED)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Device             : {device}')
    print(f'Rounds             : {NUM_ROUNDS}')
    print(f'Local epochs       : {EPOCHS}')
    print(f'Clients per round  : {CLIENTS_PER_ROUND}')
    print(f'DP clip norm       : {CLIP_NORM}')
    print(f'DP noise multiplier: {NOISE_MULTIPLIER}')
    print(f'DP noise std       : {NOISE_MULTIPLIER * CLIP_NORM:.4f}')

    print('\nLoading graphs...')
    graphs = get_all_graphs(data_dir, verbose=False)
    client_ids, _ = get_fl_partition(graphs)
    print(f'Total clients: {len(client_ids)}')

    global_model = Autoencoder().to(device)
    f1_per_round = []

    print('\nFederated Training (FedAvg AE + DP)...')
    for rnd in range(1, NUM_ROUNDS + 1):

        selected     = random.sample(client_ids, CLIENTS_PER_ROUND)
        local_states = []

        for cid in selected:
            local_model = copy.deepcopy(global_model)
            train_local_dp(local_model, graphs[cid], device, EPOCHS, LR, CLIP_NORM)
            local_states.append(local_model.state_dict())

        agg_state = fedavg_with_dp(local_states, CLIP_NORM, NOISE_MULTIPLIER)
        global_model.load_state_dict(agg_state)

        threshold = compute_threshold(global_model, graphs, device, THRESHOLD_Q)
        f1        = evaluate(global_model, graphs, threshold, device)
        f1_per_round.append(f1)

        print(f'  Round {rnd:2d}/{NUM_ROUNDS}  '
              f'clients={selected}  '
              f'F1={f1:.4f}')

    best_f1 = max(f1_per_round)
    print(f'\n{"="*50}')
    print(f'FedAvg AE + DP  —  Best weighted micro-F1 : {best_f1:.4f}')
    print(f'                   (round {f1_per_round.index(best_f1)+1})')
    print(f'{"="*50}')
    print(f'\nRound-wise F1: {[round(f,4) for f in f1_per_round]}')

    return best_f1


if __name__ == '__main__':
    DATA_DIR = "/home/user/ARYA_ANOMALY_DETECTION/anomaly_dataset-20260606T092226Z-3-001/anomaly_dataset"
    main(DATA_DIR)



Device             : cuda
Rounds             : 10
Local epochs       : 50
Clients per round  : 7
DP clip norm       : 1.0
DP noise multiplier: 0.01
DP noise std       : 0.0100

Loading graphs...
Total clients: 28

Federated Training (FedAvg AE + DP)...
  Round  1/10  clients=['3-2', '1-4', '1-1', '3-5', '2-1', '1-8', '3-4']  F1=0.8824
  Round  2/10  clients=['1-5', '3-5', '1-4', '3-3', '3-8', '3-1', '1-3']  F1=0.8860
  Round  3/10  clients=['3-10', '2-6', '1-2', '1-1', '1-3', '1-7', '1-8']  F1=0.8810
  Round  4/10  clients=['2-9', '3-11', '1-1', '3-1', '1-7', '3-4', '3-2']  F1=0.8853
  Round  5/10  clients=['3-4', '3-1', '2-6', '1-8', '2-7', '3-10', '2-1']  F1=0.8807
  Round  6/10  clients=['3-7', '1-1', '3-6', '1-6', '3-4', '2-6', '2-3']  F1=0.8807
  Round  7/10  clients=['2-1', '1-5', '1-7', '3-6', '2-3', '1-4', '1-3']  F1=0.8812
  Round  8/10  clients=['2-5', '1-4', '2-4', '3-7', '3-11', '2-1', '1-2']  F1=0.8807
  Round  9/10  clients=['3-5', '2-7', '3-1', '1-4', '2-5', '1-3', '3-7'

In [37]:
# fedavg_gae_dp.py
"""
Federated Graph Autoencoder with Differential Privacy — FedRIVER
-----------------------------------------------------------------
FL-GAE + FedAvg + Gaussian DP noise injected before aggregation.

Direct graph-aware counterpart of fedavg_ae_dp.py.
Shows DP cost on the graph model, motivating non-DP FedRIVER-GAE.

DP mechanism (Gaussian mechanism):
  1. Each client clips gradients: L2-norm ≤ CLIP_NORM
  2. Server adds Gaussian noise N(0, (NOISE_MULTIPLIER * CLIP_NORM/n)²)
     to each averaged parameter

Run:
    python fedavg_gae_dp.py --data_dir ./dataset
"""

import argparse
import random
import copy

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score

from prep_data_lib import get_all_graphs, get_fl_partition
from models import GraphAutoencoder


# ─────────────────────────────────────────────
#  Config
# ─────────────────────────────────────────────

SEED              = 42
NUM_ROUNDS        = 10
EPOCHS            = 50
LR                = 0.01
CLIENTS_PER_ROUND = 7
EXTREME_P         = 0.05
EXTREME_ALPHA     = 3.0
THRESHOLD_Q       = 0.95

# DP parameters
CLIP_NORM         = 1.0
NOISE_MULTIPLIER  = 0.01


# ─────────────────────────────────────────────
#  Reproducibility
# ─────────────────────────────────────────────

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


# ─────────────────────────────────────────────
#  Helpers
# ─────────────────────────────────────────────

def extreme_error(err: torch.Tensor, p: float = 0.05, alpha: float = 3.0):
    q = torch.quantile(err, 1 - p)
    return torch.where(err > q, err * alpha, err)


def fedavg_with_dp(local_states: list, clip_norm: float, noise_mult: float) -> dict:
    avg = copy.deepcopy(local_states[0])
    n   = len(local_states)
    for k in avg:
        for i in range(1, n):
            avg[k] = avg[k].float() + local_states[i][k].float()
        avg[k] = avg[k].float() / n
        noise_std = (noise_mult * clip_norm) / n
        avg[k]    = avg[k].float() + torch.randn_like(avg[k].float()) * noise_std
    return avg


def train_local_dp(model, data, device, epochs, lr, clip_norm):
    """Train GAE with per-step gradient clipping (DP)."""
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn   = nn.MSELoss()

    data = data.to(device)
    for _ in range(epochs):
        optimizer.zero_grad()
        x_rec, _ = model(data)
        loss = loss_fn(
            x_rec[data.train_mask],
            data.x[data.train_mask]
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
        optimizer.step()


def compute_threshold(model, graphs, device, quantile=0.95):
    model.eval()
    all_errors = []
    with torch.no_grad():
        for data in graphs.values():
            data     = data.to(device)
            x_rec, _ = model(data)
            err = torch.mean((x_rec - data.x) ** 2, dim=1)
            err = extreme_error(err, EXTREME_P, EXTREME_ALPHA)
            all_errors.append(err[data.train_mask].cpu().numpy())
    return float(np.quantile(np.concatenate(all_errors), quantile))


def evaluate(model, graphs, threshold, device):
    model.eval()
    total_nodes = 0
    weighted_f1 = 0.0
    with torch.no_grad():
        for data in graphs.values():
            data     = data.to(device)
            x_rec, _ = model(data)
            err = torch.mean((x_rec - data.x) ** 2, dim=1)
            err = extreme_error(err, EXTREME_P, EXTREME_ALPHA)

            test_mask = ~data.train_mask
            y_pred = (err[test_mask].cpu().numpy()    > threshold).astype(int)
            y_true = (data.y[test_mask].cpu().numpy() != 0).astype(int)

            f1 = f1_score(y_true, y_pred, average='micro', zero_division=0)
            weighted_f1 += data.num_nodes * f1
            total_nodes += data.num_nodes
    return weighted_f1 / total_nodes


# ─────────────────────────────────────────────
#  Main
# ─────────────────────────────────────────────

def main(data_dir: str):
    set_seed(SEED)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Device             : {device}')
    print(f'Rounds             : {NUM_ROUNDS}')
    print(f'Local epochs       : {EPOCHS}')
    print(f'Clients per round  : {CLIENTS_PER_ROUND}')
    print(f'DP clip norm       : {CLIP_NORM}')
    print(f'DP noise multiplier: {NOISE_MULTIPLIER}')

    print('\nLoading graphs...')
    graphs = get_all_graphs(data_dir, verbose=False)
    client_ids, _ = get_fl_partition(graphs)
    print(f'Total clients: {len(client_ids)}')

    global_model = GraphAutoencoder().to(device)
    f1_per_round = []

    print('\nFederated Training (FedAvg GAE + DP)...')
    for rnd in range(1, NUM_ROUNDS + 1):

        selected     = random.sample(client_ids, CLIENTS_PER_ROUND)
        local_states = []

        for cid in selected:
            local_model = copy.deepcopy(global_model)
            train_local_dp(local_model, graphs[cid], device, EPOCHS, LR, CLIP_NORM)
            local_states.append(local_model.state_dict())

        agg_state = fedavg_with_dp(local_states, CLIP_NORM, NOISE_MULTIPLIER)
        global_model.load_state_dict(agg_state)

        threshold = compute_threshold(global_model, graphs, device, THRESHOLD_Q)
        f1        = evaluate(global_model, graphs, threshold, device)
        f1_per_round.append(f1)

        print(f'  Round {rnd:2d}/{NUM_ROUNDS}  '
              f'clients={selected}  '
              f'F1={f1:.4f}')

    best_f1 = max(f1_per_round)
    print(f'\n{"="*50}')
    print(f'FedAvg GAE + DP  —  Best weighted micro-F1 : {best_f1:.4f}')
    print(f'                    (round {f1_per_round.index(best_f1)+1})')
    print(f'{"="*50}')
    print(f'\nRound-wise F1: {[round(f,4) for f in f1_per_round]}')

    return best_f1


if __name__ == '__main__':
    DATA_DIR = "/home/user/ARYA_ANOMALY_DETECTION/anomaly_dataset-20260606T092226Z-3-001/anomaly_dataset"
    main(DATA_DIR)



Device             : cuda
Rounds             : 10
Local epochs       : 50
Clients per round  : 7
DP clip norm       : 1.0
DP noise multiplier: 0.01

Loading graphs...
Total clients: 28

Federated Training (FedAvg GAE + DP)...
  Round  1/10  clients=['3-2', '1-4', '1-1', '3-5', '2-1', '1-8', '3-4']  F1=0.8808
  Round  2/10  clients=['1-5', '3-5', '1-4', '3-3', '3-8', '3-1', '1-3']  F1=0.8821
  Round  3/10  clients=['3-10', '2-6', '1-2', '1-1', '1-3', '1-7', '1-8']  F1=0.8779
  Round  4/10  clients=['2-9', '3-11', '1-1', '3-1', '1-7', '3-4', '3-2']  F1=0.8845
  Round  5/10  clients=['3-4', '3-1', '2-6', '1-8', '2-7', '3-10', '2-1']  F1=0.8791
  Round  6/10  clients=['3-7', '1-1', '3-6', '1-6', '3-4', '2-6', '2-3']  F1=0.8785
  Round  7/10  clients=['2-1', '1-5', '1-7', '3-6', '2-3', '1-4', '1-3']  F1=0.8797
  Round  8/10  clients=['2-5', '1-4', '2-4', '3-7', '3-11', '2-1', '1-2']  F1=0.8796
  Round  9/10  clients=['3-5', '2-7', '3-1', '1-4', '2-5', '1-3', '3-7']  F1=0.8806
  Round 10/10 

In [40]:
# run_all.py
"""
FedRIVER — Full Experiment Pipeline
-------------------------------------
Runs ALL baselines and FedRIVER variants in sequence.
Collects results and prints a final comparison table.

Execution order:
  ── Centralized Baselines ──────────────────────────
  1.  centralized_ae.py       (AE, no graph, no FL)
  2.  centralized_vae.py      (VAE, no graph, no FL)
  3.  centralized_gae.py      (GAE, graph, no FL)

  ── Federated Baselines (FedAvg) ───────────────────
  4.  fedavg_ae.py            (FL-AE, no graph)
  5.  fedavg_vae.py           (FL-VAE, no graph)
  6.  fedavg_gae.py           (FL-GAE, graph)

  ── Federated with Differential Privacy ────────────
  7.  fedavg_ae_dp.py         (FL-AE + DP)
  8.  fedavg_gae_dp.py        (FL-GAE + DP)

  ── FedRIVER Variants ──────────────────────────────
  9.  fedriver_gae_var.py     (Variant A — Var Reg only)
  10. fedriver_gae_kl.py      (Variant B — Var + Fixed KL)
  11. fedriver_gae_adaptive.py(Variant C — Adaptive KL)

Usage:
    python run_all.py --data_dir ./dataset

    # Run only centralized baselines
    python run_all.py --data_dir ./dataset --group centralized

    # Run only FedRIVER variants
    python run_all.py --data_dir ./dataset --group fedriver

    # Skip specific experiments
    python run_all.py --data_dir ./dataset --skip fedavg_ae_dp fedavg_gae_dp
"""

import argparse
import time
import traceback

# ─────────────────────────────────────────────
#  Import all experiment main functions
# ─────────────────────────────────────────────

from centralized_ae        import main as run_centralized_ae
from centralized_vae       import main as run_centralized_vae
from centralized_gae       import main as run_centralized_gae

from fedavg_ae             import main as run_fedavg_ae
from fedavg_vae            import main as run_fedavg_vae
from fedavg_gae            import main as run_fedavg_gae

from fedavg_ae_dp          import main as run_fedavg_ae_dp
from fedavg_gae_dp         import main as run_fedavg_gae_dp

from fedriver_gae_var      import main as run_fedriver_var
from fedriver_gae_kl       import main as run_fedriver_kl
from fedriver_gae_adaptive import main as run_fedriver_adaptive


# ─────────────────────────────────────────────
#  Experiment registry
# ─────────────────────────────────────────────

EXPERIMENTS = [
    # (name,                group,          fn)
    ('Centralized AE',      'centralized',  run_centralized_ae),
    ('Centralized VAE',     'centralized',  run_centralized_vae),
    ('Centralized GAE',     'centralized',  run_centralized_gae),
    ('FedAvg AE',           'fedavg',       run_fedavg_ae),
    ('FedAvg VAE',          'fedavg',       run_fedavg_vae),
    ('FedAvg GAE',          'fedavg',       run_fedavg_gae),
    ('FedAvg AE + DP',      'dp',           run_fedavg_ae_dp),
    ('FedAvg GAE + DP',     'dp',           run_fedavg_gae_dp),
    ('FedRIVER Var',        'fedriver',     run_fedriver_var),
    ('FedRIVER Var+KL',     'fedriver',     run_fedriver_kl),
    ('FedRIVER Adaptive',   'fedriver',     run_fedriver_adaptive),
]

# Map short skip names to experiment names
SHORT_NAMES = {
    'centralized_ae'  : 'Centralized AE',
    'centralized_vae' : 'Centralized VAE',
    'centralized_gae' : 'Centralized GAE',
    'fedavg_ae'       : 'FedAvg AE',
    'fedavg_vae'      : 'FedAvg VAE',
    'fedavg_gae'      : 'FedAvg GAE',
    'fedavg_ae_dp'    : 'FedAvg AE + DP',
    'fedavg_gae_dp'   : 'FedAvg GAE + DP',
    'fedriver_var'    : 'FedRIVER Var',
    'fedriver_kl'     : 'FedRIVER Var+KL',
    'fedriver_adaptive': 'FedRIVER Adaptive',
}


# ─────────────────────────────────────────────
#  Runner
# ─────────────────────────────────────────────

def run_all(data_dir: str, group: str = 'all', skip: list = None):
    skip = [SHORT_NAMES.get(s, s) for s in (skip or [])]

    results = {}
    timings = {}

    print('=' * 65)
    print('  FedRIVER — Full Experiment Pipeline')
    print('=' * 65)
    print(f'  Data dir : {data_dir}')
    print(f'  Group    : {group}')
    if skip:
        print(f'  Skipping : {skip}')
    print('=' * 65)

    for name, exp_group, fn in EXPERIMENTS:

        # filter by group
        if group != 'all' and exp_group != group:
            continue

        # skip list
        if name in skip:
            print(f'\n[SKIP] {name}')
            continue

        print(f'\n{"─"*65}')
        print(f'[RUN]  {name}')
        print(f'{"─"*65}')

        t0 = time.time()
        try:
            f1 = fn(data_dir)
            results[name] = f1
        except Exception as e:
            print(f'\n  *** ERROR in {name}: {e} ***')
            traceback.print_exc()
            results[name] = None
        timings[name] = time.time() - t0

        print(f'\n  → {name}  F1={results[name]:.4f}  '
              f'({timings[name]:.1f}s)')

    # ── Final summary table ────────────────────────────────────────────────
    print(f'\n\n{"="*65}')
    print(f'  RESULTS SUMMARY')
    print(f'{"="*65}')
    print(f'  {"Method":<28} {"Weighted micro-F1":>18} {"Time":>8}')
    print(f'  {"─"*55}')

    for name, _, _ in EXPERIMENTS:
        if name not in results:
            continue
        f1  = results[name]
        t   = timings.get(name, 0)
        f1_str = f'{f1:.4f}' if f1 is not None else 'ERROR'
        print(f'  {name:<28} {f1_str:>18} {t:>7.1f}s')

    print(f'{"="*65}')

    # Best result
    valid = {k: v for k, v in results.items() if v is not None}
    if valid:
        best_name = max(valid, key=valid.get)
        print(f'\n  Best: {best_name}  F1={valid[best_name]:.4f}')

    print(f'\n  Total time: {sum(timings.values()):.1f}s')
    print(f'{"="*65}')

    return results


# ─────────────────────────────────────────────
#  Main
# ─────────────────────────────────────────────

if __name__ == '__main__':
    parser = argparse.ArgumentParser(
        description='Run full FedRIVER experiment pipeline.'
    )
    parser.add_argument(
        '--data_dir', type=str, default='./dataset',
        help='Directory containing machine CSVs'
    )
    parser.add_argument(
        '--group', type=str, default='all',
        choices=['all', 'centralized', 'fedavg', 'dp', 'fedriver'],
        help='Which group of experiments to run (default: all)'
    )
    parser.add_argument(
        '--skip', nargs='+', default=[],
        help='Short names of experiments to skip (e.g. fedavg_ae_dp)'
    )
    args = parser.parse_args()

    run_all(args.data_dir, group=args.group, skip=args.skip)



ModuleNotFoundError: No module named 'centralized_ae'